<a href="https://colab.research.google.com/github/GabrielThiel/ethereum-fraud-analysis/blob/main/notebooks/02_exploracao_dataset_vagifa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:

!pip -q install kagglehub

import kagglehub
import os
import pandas as pd

# Baixar o segundo dataset
caminho = kagglehub.dataset_download(
    "vagifa/ethereum-frauddetection-dataset"
)

print("Dataset disponível em:", caminho)

# Listar os arquivos encontrados
for raiz, pastas, arquivos in os.walk(caminho):
    for arquivo in arquivos:
        print(os.path.join(raiz, arquivo))


Using Colab cache for faster access to the 'ethereum-frauddetection-dataset' dataset.
Dataset disponível em: /kaggle/input/ethereum-frauddetection-dataset
/kaggle/input/ethereum-frauddetection-dataset/.nfs000000001154195100000001
/kaggle/input/ethereum-frauddetection-dataset/transaction_dataset.csv


## 01 - Análise inicial do dataset Vagifa

O dataset Ethereum Fraud Detection Dataset (Vagifa) será analisado como
segunda fonte de dados do TCC.

Nesta etapa, serão verificadas suas dimensões, estrutura, tipos de dados,
variável-alvo e características gerais.

Também será investigada a presença de atributos que possam representar
vazamento de informação sobre o rótulo de fraude (target leakage).

Os resultados serão posteriormente comparados com os do ETFD, respeitando
as diferenças entre as unidades de análise dos datasets.

In [37]:

import pandas as pd
import os

# Caminho do dataset baixado pelo KaggleHub
arquivo = os.path.join(caminho, "transaction_dataset.csv")

# Carregar o dataset
df_vagifa = pd.read_csv(arquivo)

print("=== DIMENSÕES ===")
print("Linhas:", df_vagifa.shape[0])
print("Colunas:", df_vagifa.shape[1])

print("\n=== COLUNAS ===")
print(df_vagifa.columns.tolist())

print("\n=== TIPOS DE DADOS ===")
print(df_vagifa.dtypes.to_string())

print("\n=== PRIMEIROS 5 REGISTROS ===")
display(df_vagifa.head())


=== DIMENSÕES ===
Linhas: 9841
Colunas: 51

=== COLUNAS ===
['Unnamed: 0', 'Index', 'Address', 'FLAG', 'Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received ', 'avg val received', 'min val sent', 'max val sent', 'avg val sent', 'min value sent to contract', 'max val sent to contract', 'avg value sent to contract', 'total transactions (including tnx to create contract', 'total Ether sent', 'total ether received', 'total ether sent contracts', 'total ether balance', ' Total ERC20 tnxs', ' ERC20 total Ether received', ' ERC20 total ether sent', ' ERC20 total Ether sent contract', ' ERC20 uniq sent addr', ' ERC20 uniq rec addr', ' ERC20 uniq sent addr.1', ' ERC20 uniq rec contract addr', ' ERC20 avg time between sent tnx', ' ERC20 avg time between rec tnx', ' ERC20 avg time between

,Unnamed: 0,Index,Address,FLAG,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,...,ERC20 min val sent,ERC20 max val sent,ERC20 avg val sent,ERC20 min val sent contract,ERC20 max val sent contract,ERC20 avg val sent contract,ERC20 uniq sent token name,ERC20 uniq rec token name,ERC20 most sent token type,ERC20_most_rec_token_type
0,0,1,0x00009277775ac7d0d59eaad8fee3d10ac6c805e8,0,844.26,1093.71,704785.63,721,89,0,...,0.000000,1.683100e+07,271779.920000,0.0,0.0,0.0,39.0,57.0,Cofoundit,Numeraire
1,1,2,0x0002b44ddb1476db43c868bd494422ee4c136fed,0,12709.07,2958.44,1218216.73,94,8,0,...,2.260809,2.260809e+00,2.260809,0.0,0.0,0.0,1.0,7.0,Livepeer Token,Livepeer Token
2,2,3,0x0002bda54cb772d040f779e88eb453cac0daa244,0,246194.54,2434.02,516729.30,2,10,0,...,0.000000,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,8.0,NaN,XENON
3,3,4,0x00038e6ba2fd5c09aedb96697c8d7b8fa6632e5e,0,10219.60,15785.09,397555.90,25,9,0,...,100.000000,9.029231e+03,3804.076893,0.0,0.0,0.0,1.0,11.0,Raiden,XENON
4,4,5,0x00062d1dd1afb6fb02540ddad9cdebfe568e0d89,0,36.61,10707.77,382472.42,4598,20,1,...,0.000000,4.500000e+04,13726.659220,0.0,0.0,0.0,6.0,27.0,StatusNetwork,EOS


## 02 - Investigação da variável-alvo e identificadores

Nesta etapa, será analisada a distribuição da variável-alvo FLAG e sua
possível relação com as colunas de índice presentes no dataset.

O objetivo é verificar se a organização dos registros ou os
identificadores podem fornecer informações sobre a classificação de
fraude, representando um possível risco de target leakage.

Nenhuma feature será removida antes da análise dos resultados.

In [38]:

# ==========================================
# Distribuição das classes e análise dos índices
# ==========================================

print("=== DIMENSÕES ===")
print(df_vagifa.shape)

print("\n=== DISTRIBUIÇÃO DE FLAG ===")
print(df_vagifa["FLAG"].value_counts())

print("\n=== PERCENTUAL POR CLASSE ===")
print(
    (df_vagifa["FLAG"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\n=== PRIMEIROS 10 REGISTROS ===")
display(
    df_vagifa[
        ["Unnamed: 0", "Index", "FLAG"]
    ].head(10)
)

print("\n=== ÚLTIMOS 10 REGISTROS ===")
display(
    df_vagifa[
        ["Unnamed: 0", "Index", "FLAG"]
    ].tail(10)
)

print("\n=== DISTRIBUIÇÃO DOS ÍNDICES POR CLASSE ===")
display(
    df_vagifa.groupby("FLAG")[
        ["Unnamed: 0", "Index"]
    ].agg(["min", "max", "mean"])
)

print("\n=== FRAUDE POR FAIXA DO ÍNDICE ===")

df_analise_indice = df_vagifa.copy()

df_analise_indice["faixa_indice"] = pd.qcut(
    df_analise_indice["Index"],
    q=10,
    duplicates="drop"
)

display(
    df_analise_indice.groupby(
        "faixa_indice", observed=True
    )["FLAG"].agg(
        quantidade="count",
        fraudes="sum",
        proporcao_fraude="mean"
    )
)


=== DIMENSÕES ===
(9841, 51)

=== DISTRIBUIÇÃO DE FLAG ===
FLAG
0    7662
1    2179
Name: count, dtype: int64

=== PERCENTUAL POR CLASSE ===
FLAG
0    77.86
1    22.14
Name: proportion, dtype: float64

=== PRIMEIROS 10 REGISTROS ===


,Unnamed: 0,Index,FLAG
0,0,1,0
1,1,2,0
2,2,3,0
3,3,4,0
4,4,5,0
5,5,6,0
6,6,7,0
7,7,8,0
8,8,9,0
9,9,10,0



=== ÚLTIMOS 10 REGISTROS ===


,Unnamed: 0,Index,FLAG
9831,9831,2170,1
9832,9832,2171,1
9833,9833,2172,1
9834,9834,2173,1
9835,9835,2174,1
9836,9836,2175,1
9837,9837,2176,1
9838,9838,2177,1
9839,9839,2178,1
9840,9840,2179,1



=== DISTRIBUIÇÃO DOS ÍNDICES POR CLASSE ===


Unnamed: 0               Index                   
            min   max    mean   min   max         mean
FLAG                                                  
0             0  7661  3830.5     1  4729  2021.247194
1          7662  9840  8751.0     1  2179  1090.000000


=== FRAUDE POR FAIXA DO ÍNDICE ===


,quantidade,fraudes,proporcao_fraude
faixa_indice,,,
"(0.999, 329.0]",987,329,0.333333
"(329.0, 657.0]",984,328,0.333333
"(657.0, 985.0]",984,328,0.333333
"(985.0, 1313.0]",984,328,0.333333
"(1313.0, 1641.0]",984,328,0.333333
"(1641.0, 1969.0]",984,328,0.333333
"(1969.0, 2355.0]",982,210,0.213849
"(2355.0, 2847.0]",984,0,0.000000
"(2847.0, 3745.0]",984,0,0.000000


In [4]:

# ==========================================
# Auditoria de possível target leakage
# ==========================================

print("=== DISTRIBUIÇÃO DAS CLASSES ===")
print(df_vagifa["FLAG"].value_counts())

print("\n=== REGRA BASEADA EM Unnamed: 0 ===")

predicao_indice = (
    df_vagifa["Unnamed: 0"] >= 7662
).astype(int)

acertos = (predicao_indice == df_vagifa["FLAG"]).sum()
erros = (predicao_indice != df_vagifa["FLAG"]).sum()

print("Acertos:", acertos)
print("Erros:", erros)
print("Proporção de acertos:", acertos / len(df_vagifa))

print("\n=== ANÁLISE DE Index ===")

resumo_index = (
    df_vagifa.groupby("Index")["FLAG"]
    .agg(["count", "nunique", "min", "max"])
)

print(
    "Valores de Index presentes nas duas classes:",
    (resumo_index["nunique"] == 2).sum()
)

print(
    "Valores de Index exclusivos da classe 0:",
    (
        (resumo_index["min"] == 0) &
        (resumo_index["max"] == 0)
    ).sum()
)

print(
    "Valores de Index exclusivos da classe 1:",
    (
        (resumo_index["min"] == 1) &
        (resumo_index["max"] == 1)
    ).sum()
)


=== DISTRIBUIÇÃO DAS CLASSES ===
FLAG
0    7662
1    2179
Name: count, dtype: int64

=== REGRA BASEADA EM Unnamed: 0 ===
Acertos: 9841
Erros: 0
Proporção de acertos: 1.0

=== ANÁLISE DE Index ===
Valores de Index presentes nas duas classes: 2179
Valores de Index exclusivos da classe 0: 2550
Valores de Index exclusivos da classe 1: 0



## 03 - Análise de qualidade dos dados

Após confirmar que a coluna `Unnamed: 0` reproduz
integralmente os rótulos de fraude, será realizada
uma análise da qualidade do dataset Vagifa.

Serão investigados valores ausentes, duplicatas,
endereços repetidos, conflitos de classificação
e atributos constantes.

O DataFrame original será preservado, permitindo
documentar as decisões de limpeza antes de
alterar os dados.


In [5]:

# ==========================================
# Auditoria de qualidade - Dataset Vagifa
# ==========================================

print("=== DIMENSÕES ===")
print(df_vagifa.shape)

print("\n=== VALORES AUSENTES ===")
nulos = df_vagifa.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print("Colunas com valores ausentes:", len(nulos))
display(nulos.to_frame("Quantidade_nulos"))

print("\n=== LINHAS DUPLICADAS ===")
print("Duplicatas exatas:", df_vagifa.duplicated().sum())

print("\n=== ENDEREÇOS ===")
print("Endereços únicos:", df_vagifa["Address"].nunique())

print(
    "Linhas com endereços repetidos:",
    df_vagifa.duplicated(
        subset=["Address"], keep=False
    ).sum()
)

print("\n=== CONFLITOS DE RÓTULO POR ENDEREÇO ===")

conflitos = (
    df_vagifa.groupby("Address")["FLAG"]
    .nunique()
)

print(
    "Endereços associados às duas classes:",
    (conflitos > 1).sum()
)

print("\n=== FEATURES CONSTANTES ===")

constantes = df_vagifa.columns[
    df_vagifa.nunique(dropna=False) <= 1
].tolist()

print(constantes)

print("\n=== COLUNAS NÃO NUMÉRICAS ===")
print(
    df_vagifa.select_dtypes(
        exclude="number"
    ).columns.tolist()
)


=== DIMENSÕES ===
(9841, 51)

=== VALORES AUSENTES ===
Colunas com valores ausentes: 25


,Quantidade_nulos
ERC20 most sent token type,2697
ERC20_most_rec_token_type,871
ERC20 total ether sent,829
ERC20 total Ether received,829
ERC20 uniq sent addr,829
ERC20 uniq rec addr,829
ERC20 uniq sent addr.1,829
ERC20 total Ether sent contract,829
Total ERC20 tnxs,829
ERC20 avg time between sent tnx,829



=== LINHAS DUPLICADAS ===
Duplicatas exatas: 0

=== ENDEREÇOS ===
Endereços únicos: 9816
Linhas com endereços repetidos: 50

=== CONFLITOS DE RÓTULO POR ENDEREÇO ===
Endereços associados às duas classes: 0

=== FEATURES CONSTANTES ===
[]

=== COLUNAS NÃO NUMÉRICAS ===
['Address', ' ERC20 most sent token type', ' ERC20_most_rec_token_type']


## 04 - Investigação de valores ausentes e endereços repetidos

Foram identificados valores ausentes em diferentes atributos
relacionados a tokens ERC-20 e registros com endereços repetidos.

Nesta etapa, será investigada a distribuição dos valores ausentes
entre as classes e verificado se os registros associados ao mesmo
endereço possuem características diferentes.

Nenhuma imputação ou remoção será realizada antes da análise.

In [6]:

# ==========================================
# Análise de valores ausentes
# ==========================================

colunas_nulas = df_vagifa.columns[
    df_vagifa.isna().any()
].tolist()

print("Colunas com valores ausentes:", len(colunas_nulas))

df_analise_nulos = df_vagifa.copy()

df_analise_nulos["tem_nulos"] = (
    df_vagifa[colunas_nulas].isna().any(axis=1)
)

print("\n=== VALORES AUSENTES POR CLASSE ===")

resumo_nulos = df_analise_nulos.groupby("FLAG")[
    "tem_nulos"
].agg(
    total="count",
    registros_com_nulos="sum",
    proporcao="mean"
)

resumo_nulos["proporcao"] *= 100
display(resumo_nulos)

print("\n=== QUANTIDADE DE NULOS POR REGISTRO ===")

quantidade_nulos = (
    df_vagifa[colunas_nulas].isna().sum(axis=1)
)

display(
    quantidade_nulos.value_counts().sort_index()
)

# ==========================================
# Investigação de endereços repetidos
# ==========================================

print("\n=== DUPLICATAS IGNORANDO ÍNDICES ===")

colunas_sem_indices = [
    c for c in df_vagifa.columns
    if c not in ["Unnamed: 0", "Index"]
]

print(
    df_vagifa.duplicated(
        subset=colunas_sem_indices
    ).sum()
)

print("\n=== DIFERENÇAS ENTRE ENDEREÇOS REPETIDOS ===")

repetidos = df_vagifa[
    df_vagifa.duplicated(
        subset=["Address"],
        keep=False
    )
].copy()

features_comparacao = [
    c for c in df_vagifa.columns
    if c not in [
        "Unnamed: 0", "Index", "Address", "FLAG"
    ]
]

diferencas = (
    repetidos.groupby("Address")[features_comparacao]
    .nunique(dropna=False)
    .gt(1)
    .sum(axis=1)
)

print(
    "Endereços com características diferentes:",
    (diferencas > 0).sum()
)

print(
    "Endereços com características idênticas:",
    (diferencas == 0).sum()
)

display(
    diferencas.to_frame(
        "quantidade_features_diferentes"
    ).head(10)
)


Colunas com valores ausentes: 25

=== VALORES AUSENTES POR CLASSE ===


,total,registros_com_nulos,proporcao
FLAG,,,
0,7662,1891,24.680240
1,2179,829,38.044975



=== QUANTIDADE DE NULOS POR REGISTRO ===


,count
0,7121
1,1872
2,19
25,829



=== DUPLICATAS IGNORANDO ÍNDICES ===
18

=== DIFERENÇAS ENTRE ENDEREÇOS REPETIDOS ===
Endereços com características diferentes: 7
Endereços com características idênticas: 18


,quantidade_features_diferentes
Address,
0x4c13f6966dc24c92489344f0fd6f0e61f3489b84,0
0x4c1da8781f6ca312bc11217b3f61e5dfdf428de1,0
0x4c24af967901ec87a6644eb1ef42b680f58e67f5,1
0x4c268c7b1d51b369153d6f1f28c61b15f0e17746,0
0x4c26a3c12a64f33a3546fbb206c5365ce8e82c20,0
0x4c271764eadcf0d07e5a937b2de290294c9d11c2,1
0x4c27438a77738153f6cf3ed890b2817d52ebf584,0
0x4c391cc032c9107b596267610a05262c90fc2df7,0
0x4c4a03e100b4b104355edc4c50ce12b9a2879547,0



## 05 - Relação entre valores ausentes e a variável FLAG

A análise de qualidade identificou 2.720 registros com pelo menos
um valor ausente, incluindo 829 registros com 25 atributos ausentes.

A proporção de registros com dados ausentes também apresentou
diferenças entre as classes FLAG = 0 e FLAG = 1.

Será investigado se os padrões de ausência de dados estão associados
à classificação, representando um possível atalho de aprendizado
para os modelos.

Não serão realizadas remoções ou imputações até que essa relação
seja analisada.


In [7]:

# ==========================================
# Relação entre quantidade de nulos e FLAG
# ==========================================

df_auditoria_nulos = df_vagifa.copy()

df_auditoria_nulos["qtd_nulos"] = (
    df_vagifa.isna().sum(axis=1)
)

print("=== QUANTIDADE DE NULOS POR CLASSE ===")

tabela_nulos_flag = pd.crosstab(
    df_auditoria_nulos["qtd_nulos"],
    df_auditoria_nulos["FLAG"],
    margins=True
)

display(tabela_nulos_flag)

print("\n=== PERCENTUAL DE FRAUDE POR QUANTIDADE DE NULOS ===")

resumo_nulos_flag = (
    df_auditoria_nulos
    .groupby("qtd_nulos")["FLAG"]
    .agg(
        registros="count",
        fraudes="sum",
        percentual_fraude="mean"
    )
)

resumo_nulos_flag["percentual_fraude"] *= 100

display(resumo_nulos_flag)


=== QUANTIDADE DE NULOS POR CLASSE ===


FLAG,0,1,All
qtd_nulos,,,
0,5771,1350,7121
1,1872,0,1872
2,19,0,19
25,0,829,829
All,7662,2179,9841



=== PERCENTUAL DE FRAUDE POR QUANTIDADE DE NULOS ===


,registros,fraudes,percentual_fraude
qtd_nulos,,,
0,7121,1350,18.958012
1,1872,0,0.000000
2,19,0,0.000000
25,829,829,100.000000


In [8]:

# ==========================================
# Investigação dos padrões de ausência
# ==========================================

qtd_nulos = df_vagifa.isna().sum(axis=1)

for quantidade in [1, 2, 25]:

    grupo = df_vagifa.loc[qtd_nulos == quantidade]

    print(f"\n=== REGISTROS COM {quantidade} NULOS ===")
    print("Quantidade de registros:", len(grupo))

    print("\nDistribuição de FLAG:")
    print(grupo["FLAG"].value_counts())

    print("\nColunas com valores ausentes:")
    colunas_ausentes = grupo.isna().sum()
    colunas_ausentes = colunas_ausentes[
        colunas_ausentes > 0
    ].sort_values(ascending=False)

    print(colunas_ausentes.to_string())

print("\n=== ATRIBUTOS ERC-20 ===")

colunas_erc20 = [
    c for c in df_vagifa.columns
    if "ERC20" in c.upper()
]

print("Quantidade de colunas ERC-20:", len(colunas_erc20))
print("Colunas ERC-20:", colunas_erc20)



=== REGISTROS COM 1 NULOS ===
Quantidade de registros: 1872

Distribuição de FLAG:
FLAG
0    1872
Name: count, dtype: int64

Colunas com valores ausentes:
ERC20 most sent token type    1849
ERC20_most_rec_token_type       23

=== REGISTROS COM 2 NULOS ===
Quantidade de registros: 19

Distribuição de FLAG:
FLAG
0    19
Name: count, dtype: int64

Colunas com valores ausentes:
ERC20 most sent token type    19
ERC20_most_rec_token_type     19

=== REGISTROS COM 25 NULOS ===
Quantidade de registros: 829

Distribuição de FLAG:
FLAG
1    829
Name: count, dtype: int64

Colunas com valores ausentes:
Total ERC20 tnxs                       829
ERC20 total Ether received             829
ERC20 total ether sent                 829
ERC20 total Ether sent contract        829
ERC20 uniq sent addr                   829
ERC20 uniq rec addr                    829
ERC20 uniq sent addr.1                 829
ERC20 uniq rec contract addr           829
ERC20 avg time between sent tnx        829
ERC20 avg time


## 06 - Investigação dos endereços repetidos

A auditoria identificou 25 endereços repetidos no dataset.

Desconsiderando as colunas de índice, 18 desses endereços
apresentam registros com características idênticas,
enquanto 7 apresentam diferenças em pelo menos uma feature.

Nenhum endereço apresentou conflito entre os rótulos
de fraude.

Nesta etapa, serão investigadas as diferenças entre
os registros repetidos antes de definir o procedimento
de limpeza.

O objetivo é evitar a remoção indevida de informações
e impedir que o mesmo endereço apareça nos conjuntos
de treinamento e teste.


In [9]:

# ==========================================
# Investigar endereços com características diferentes
# ==========================================

df_auditoria = df_vagifa.copy()

# Normalizar os endereços para comparação
df_auditoria["Address_norm"] = (
    df_auditoria["Address"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Identificar endereços repetidos
repetidos = df_auditoria[
    df_auditoria.duplicated(
        subset=["Address_norm"],
        keep=False
    )
].copy()

# Colunas utilizadas na comparação
colunas_analise = [
    c for c in df_auditoria.columns
    if c not in [
        "Unnamed: 0",
        "Index",
        "Address",
        "Address_norm",
        "FLAG"
    ]
]

print("=== ENDEREÇOS REPETIDOS ===")
print("Registros:", len(repetidos))
print("Endereços:", repetidos["Address_norm"].nunique())

print("\n=== ENDEREÇOS COM DIFERENÇAS ===")

for endereco, grupo in repetidos.groupby("Address_norm"):

    colunas_diferentes = [
        c for c in colunas_analise
        if grupo[c].nunique(dropna=False) > 1
    ]

    if colunas_diferentes:

        print("\nEndereço:", endereco)
        print("Registros:", len(grupo))
        print("Features diferentes:", colunas_diferentes)

        display(
            grupo[
                ["FLAG"] + colunas_diferentes
            ]
        )


=== ENDEREÇOS REPETIDOS ===
Registros: 50
Endereços: 25

=== ENDEREÇOS COM DIFERENÇAS ===

Endereço: 0x4c24af967901ec87a6644eb1ef42b680f58e67f5
Registros: 2
Features diferentes: [' ERC20_most_rec_token_type']


,FLAG,ERC20_most_rec_token_type
2910,0,INS Promo
2935,0,VIU



Endereço: 0x4c271764eadcf0d07e5a937b2de290294c9d11c2
Registros: 2
Features diferentes: [' ERC20_most_rec_token_type']


,FLAG,ERC20_most_rec_token_type
2913,0,DATAcoin
2938,0,OmiseGO



Endereço: 0x4c7520df888aa4569a37ac7d132f89c65821f0af
Registros: 2
Features diferentes: [' ERC20_most_rec_token_type']


,FLAG,ERC20_most_rec_token_type
2919,0,XENON
2944,0,Livepeer Token


... (4 outputs left out)


Endereço: 0x4cad652b71519a7a68d05dada31122c4c9a5ed95
Registros: 2
Features diferentes: [' ERC20 most sent token type']


,FLAG,ERC20 most sent token type
2924,0,Dao.Casino
2949,0,AVT



Endereço: 0x4cb981a7a2956cdfa8afbf454cb4e1b13c69aeb9
Registros: 2
Features diferentes: [' ERC20_most_rec_token_type']


,FLAG,ERC20_most_rec_token_type
2926,0,Livepeer Token
2951,0,DATAcoin


In [10]:

# ==========================================
# Auditoria final dos endereços duplicados
# ==========================================

df_auditoria = df_vagifa.copy()

# Remover espaços extras dos nomes das colunas
df_auditoria.columns = df_auditoria.columns.str.strip()

# Normalizar os endereços
df_auditoria["Address_norm"] = (
    df_auditoria["Address"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Identificar registros de endereços repetidos
repetidos = df_auditoria[
    df_auditoria.duplicated(
        subset=["Address_norm"],
        keep=False
    )
].copy()

# Features numéricas, sem identificadores e rótulo
colunas_numericas = [
    c for c in df_auditoria.select_dtypes(
        include="number"
    ).columns
    if c not in ["Unnamed: 0", "Index", "FLAG"]
]

# Features categóricas de tokens
colunas_tokens = [
    "ERC20 most sent token type",
    "ERC20_most_rec_token_type"
]

resultados_duplicatas = []

for endereco, grupo in repetidos.groupby("Address_norm"):

    diferentes_numericas = [
        c for c in colunas_numericas
        if grupo[c].nunique(dropna=False) > 1
    ]

    diferentes_tokens = [
        c for c in colunas_tokens
        if grupo[c].nunique(dropna=False) > 1
    ]

    resultados_duplicatas.append({
        "Address": endereco,
        "Registros": len(grupo),
        "Diferencas_numericas": len(diferentes_numericas),
        "Diferencas_tokens": len(diferentes_tokens),
        "Conflito_FLAG": grupo["FLAG"].nunique() > 1
    })

df_resumo_duplicatas = pd.DataFrame(resultados_duplicatas)

display(df_resumo_duplicatas)

print("\n=== RESUMO ===")

print(
    "Endereços repetidos:",
    len(df_resumo_duplicatas)
)

print(
    "Endereços com diferenças numéricas:",
    (df_resumo_duplicatas["Diferencas_numericas"] > 0).sum()
)

print(
    "Endereços com diferenças nos tokens:",
    (df_resumo_duplicatas["Diferencas_tokens"] > 0).sum()
)

print(
    "Endereços com conflito de FLAG:",
    df_resumo_duplicatas["Conflito_FLAG"].sum()
)


,Address,Registros,Diferencas_numericas,Diferencas_tokens,Conflito_FLAG
0,0x4c13f6966dc24c92489344f0fd6f0e61f3489b84,2,0,0,False
1,0x4c1da8781f6ca312bc11217b3f61e5dfdf428de1,2,0,0,False
2,0x4c24af967901ec87a6644eb1ef42b680f58e67f5,2,0,1,False
3,0x4c268c7b1d51b369153d6f1f28c61b15f0e17746,2,0,0,False
4,0x4c26a3c12a64f33a3546fbb206c5365ce8e82c20,2,0,0,False
5,0x4c271764eadcf0d07e5a937b2de290294c9d11c2,2,0,1,False
6,0x4c27438a77738153f6cf3ed890b2817d52ebf584,2,0,0,False
7,0x4c391cc032c9107b596267610a05262c90fc2df7,2,0,0,False
8,0x4c4a03e100b4b104355edc4c50ce12b9a2879547,2,0,0,False
9,0x4c4da560350e302232a184c8fa16b126a772c326,2,0,0,False



=== RESUMO ===
Endereços repetidos: 25
Endereços com diferenças numéricas: 0
Endereços com diferenças nos tokens: 7
Endereços com conflito de FLAG: 0


## 07 - Preparação da base para modelagem

Após a auditoria, verificou-se que os 25 endereços repetidos
não apresentaram diferenças numéricas nem conflitos de FLAG.

Para os experimentos, será mantido um registro por endereço
Ethereum normalizado.

As colunas artificiais de índice serão excluídas devido ao
risco confirmado de vazamento de informação.

Os nomes categóricos dos tokens ERC-20 serão inicialmente
excluídos da versão de modelagem, pois apresentaram
inconsistências entre registros de um mesmo endereço.

Também será criada uma versão sem atributos ERC-20 para
investigar o desempenho sem o padrão de valores ausentes
fortemente associado à classe fraudulenta.

O dataset original será preservado.

In [11]:

# ==========================================
# Preparação inicial do dataset Vagifa
# ==========================================

# Preservar o DataFrame original
df_vagifa_preparado = df_vagifa.copy()

# Padronizar nomes das colunas
df_vagifa_preparado.columns = (
    df_vagifa_preparado.columns.str.strip()
)

# Normalizar endereços Ethereum
df_vagifa_preparado["Address_norm"] = (
    df_vagifa_preparado["Address"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Remover índices e campos categóricos
# inconsistentes na auditoria
colunas_remover = [
    "Unnamed: 0",
    "Index",
    "Address",
    "ERC20 most sent token type",
    "ERC20_most_rec_token_type"
]

df_vagifa_preparado = (
    df_vagifa_preparado
    .drop(columns=colunas_remover)
)

# Manter um registro por endereço
df_vagifa_preparado = (
    df_vagifa_preparado
    .drop_duplicates(
        subset=["Address_norm"],
        keep="first"
    )
    .reset_index(drop=True)
)

# Criar cenário sem os atributos ERC-20
colunas_erc20 = [
    c for c in df_vagifa_preparado.columns
    if "erc20" in c.lower()
]

df_vagifa_sem_erc20 = (
    df_vagifa_preparado
    .drop(columns=colunas_erc20)
    .copy()
)

# Separar entradas e alvo
# Address_norm permanece apenas para auditoria
X_vagifa = df_vagifa_sem_erc20.drop(
    columns=["FLAG", "Address_norm"]
)

y_vagifa = df_vagifa_sem_erc20["FLAG"]

print("=== DATASET ORIGINAL ===")
print(df_vagifa.shape)

print("\n=== DATASET PREPARADO ===")
print(df_vagifa_preparado.shape)

print("\n=== DATASET SEM ERC-20 ===")
print(df_vagifa_sem_erc20.shape)

print("\n=== FEATURES NUMÉRICAS ===")
print(X_vagifa.columns.tolist())
print("Quantidade:", X_vagifa.shape[1])

print("\n=== DISTRIBUIÇÃO DE FLAG ===")
print(y_vagifa.value_counts())
print((y_vagifa.value_counts(normalize=True) * 100).round(2))

print("\n=== VERIFICAÇÃO DE QUALIDADE ===")
print("Valores nulos:", X_vagifa.isna().sum().sum())
print(
    "Endereços repetidos:",
    df_vagifa_sem_erc20["Address_norm"].duplicated().sum()
)
print(
    "Features não numéricas:",
    X_vagifa.select_dtypes(exclude="number").columns.tolist()
)


=== DATASET ORIGINAL ===
(9841, 51)

=== DATASET PREPARADO ===
(9816, 47)

=== DATASET SEM ERC-20 ===
(9816, 24)

=== FEATURES NUMÉRICAS ===
['Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received', 'avg val received', 'min val sent', 'max val sent', 'avg val sent', 'min value sent to contract', 'max val sent to contract', 'avg value sent to contract', 'total transactions (including tnx to create contract', 'total Ether sent', 'total ether received', 'total ether sent contracts', 'total ether balance']
Quantidade: 22

=== DISTRIBUIÇÃO DE FLAG ===
FLAG
0    7637
1    2179
Name: count, dtype: int64
FLAG
0    77.8
1    22.2
Name: proportion, dtype: float64

=== VERIFICAÇÃO DE QUALIDADE ===
Valores nulos: 0
Endereços repetidos: 0
Features não numéricas: []



## 08 - Auditoria das features numéricas

Após a preparação inicial, foram obtidos 9.816 registros
e 22 features numéricas, excluindo os atributos ERC-20.

Nesta etapa, serão analisados os valores mínimos e máximos,
a quantidade de valores distintos, a presença de valores
negativos ou infinitos e a frequência de zeros.

Também será investigada a distribuição dos valores zero
entre as classes FLAG = 0 e FLAG = 1.

O objetivo é identificar possíveis inconsistências e
atributos que apresentem associações muito fortes com
a variável-alvo.

Nenhuma feature será removida automaticamente.


In [12]:

import numpy as np
import pandas as pd

# ==========================================
# Auditoria das 22 features - Vagifa
# ==========================================

resumo_features = pd.DataFrame(
    index=X_vagifa.columns
)

resumo_features["Tipo"] = (
    X_vagifa.dtypes.astype(str)
)

resumo_features["Valores_unicos"] = (
    X_vagifa.nunique()
)

resumo_features["Minimo"] = (
    X_vagifa.min()
)

resumo_features["Mediana"] = (
    X_vagifa.median()
)

resumo_features["Maximo"] = (
    X_vagifa.max()
)

resumo_features["Negativos"] = (
    (X_vagifa < 0).sum()
)

resumo_features["Infinitos"] = (
    np.isinf(
        X_vagifa.to_numpy(dtype=float)
    ).sum(axis=0)
)

# Percentual de zeros em toda a base
resumo_features["Zeros_geral_%"] = (
    (X_vagifa == 0).mean() * 100
).round(2)

# Percentual de zeros para FLAG = 0
resumo_features["Zeros_FLAG_0_%"] = (
    (X_vagifa.loc[y_vagifa == 0] == 0)
    .mean() * 100
).round(2)

# Percentual de zeros para FLAG = 1
resumo_features["Zeros_FLAG_1_%"] = (
    (X_vagifa.loc[y_vagifa == 1] == 0)
    .mean() * 100
).round(2)

# Diferença absoluta entre as classes
resumo_features["Diferenca_zeros_pp"] = (
    resumo_features["Zeros_FLAG_1_%"]
    - resumo_features["Zeros_FLAG_0_%"]
).abs().round(2)

print("=== AUDITORIA DAS 22 FEATURES ===")

display(
    resumo_features.sort_values(
        "Diferenca_zeros_pp",
        ascending=False
    )
)

print("\n=== RESUMO ===")

print(
    "Features analisadas:",
    len(resumo_features)
)

print(
    "Valores negativos:",
    resumo_features["Negativos"].sum()
)

print(
    "Valores infinitos:",
    resumo_features["Infinitos"].sum()
)


=== AUDITORIA DAS 22 FEATURES ===


,Tipo,Valores_unicos,Minimo,Mediana,Maximo,Negativos,Infinitos,Zeros_geral_%,Zeros_FLAG_0_%,Zeros_FLAG_1_%,Diferenca_zeros_pp
Avg min between sent tnx,float64,5013,0.00,17.335000,4.302877e+05,0,0,35.80,27.04,66.50,39.46
max value received,float64,6302,0.00,6.000000,8.000000e+05,0,0,7.20,0.65,30.15,29.50
total ether received,float64,6728,0.00,30.493969,2.858159e+07,0,0,7.20,0.65,30.15,29.50
avg val received,float64,6767,0.00,1.727367,2.836188e+05,0,0,7.22,0.68,30.15,29.47
total ether balance,float64,5717,-15605352.04,0.001722,1.428864e+07,976,0,7.09,0.55,30.01,29.46
Time Diff between first and last (Mins),float64,7810,0.00,46569.805000,1.954861e+06,0,0,6.41,0.60,26.76,26.16
Unique Received From Addresses,int64,256,0.00,2.000000,9.999000e+03,0,0,5.61,0.13,24.83,24.70
Received Tnx,int64,727,0.00,4.000000,1.000000e+04,0,0,5.61,0.13,24.83,24.70
total transactions (including tnx to create contract,int64,897,0.00,8.000000,1.999500e+04,0,0,5.39,0.00,24.28,24.28
Sent tnx,int64,641,0.00,3.000000,1.000000e+04,0,0,20.48,15.57,37.68,22.11



=== RESUMO ===
Features analisadas: 22
Valores negativos: 976
Valores infinitos: 0



## 09 - Investigação dos valores negativos

A auditoria das 22 features identificou 976 valores
negativos na variável `total ether balance`.

Será investigado se esses valores correspondem à
diferença entre o total de Ether recebido e enviado.

Também será analisada a distribuição dos registros
com saldo negativo entre as classes FLAG = 0 e FLAG = 1.

Nenhum registro será removido antes da investigação.


In [13]:

import numpy as np
import pandas as pd

# ==========================================
# Investigação do saldo negativo
# ==========================================

col_balance = "total ether balance"
col_received = "total ether received"
col_sent = "total Ether sent"

# Registros com saldo negativo
saldo_negativo = X_vagifa[col_balance] < 0

print("=== SALDOS NEGATIVOS ===")
print("Quantidade:", saldo_negativo.sum())

print("\n=== DISTRIBUIÇÃO POR CLASSE ===")

tabela_negativos = pd.crosstab(
    y_vagifa,
    saldo_negativo,
    rownames=["FLAG"],
    colnames=["Saldo negativo"]
)

display(tabela_negativos)

print("\n=== PERCENTUAL POR CLASSE ===")

display(
    pd.crosstab(
        y_vagifa,
        saldo_negativo,
        normalize="index"
    ).mul(100).round(2)
)

print("\n=== VERIFICAÇÃO DO CÁLCULO DO SALDO ===")

# Hipótese: saldo = recebido - enviado
saldo_calculado = (
    X_vagifa[col_received] -
    X_vagifa[col_sent]
)

saldo_registrado = X_vagifa[col_balance]

diferenca = (saldo_registrado - saldo_calculado).abs()

print(
    "Registros com diferença superior a 0,01 ETH:",
    (diferenca > 0.01).sum()
)

print("Maior diferença encontrada:", diferenca.max())

print("\n=== EXEMPLOS DE SALDOS NEGATIVOS ===")

display(
    X_vagifa.loc[
        saldo_negativo,
        [col_received, col_sent, col_balance]
    ].head(10)
)


=== SALDOS NEGATIVOS ===
Quantidade: 976

=== DISTRIBUIÇÃO POR CLASSE ===


Saldo negativo,False,True
FLAG,,
0,6809,828
1,2031,148



=== PERCENTUAL POR CLASSE ===


total ether balance,False,True
FLAG,,
0,89.16,10.84
1,93.21,6.79



=== VERIFICAÇÃO DO CÁLCULO DO SALDO ===
Registros com diferença superior a 0,01 ETH: 2
Maior diferença encontrada: 0.046028712999999666

=== EXEMPLOS DE SALDOS NEGATIVOS ===


,total ether received,total Ether sent,total ether balance
0,586.466675,865.691093,-279.224419
1,3.085478,3.087297,-0.001819
3,895.399559,1750.045862,-854.646303
4,53.421897,104.318883,-50.896986
7,4.455490,8.703392,-4.247902
9,50.100000,149.077000,-98.977000
14,1851.006376,3273.988723,-1422.982347
17,182.211566,257.307000,-75.095434
21,58.850704,157.092350,-98.241647
26,35.000000,62.341000,-27.341000


In [14]:

# ==========================================
# Auditoria das diferenças no cálculo do saldo
# ==========================================

df_verificacao_saldo = X_vagifa[
    [
        "total ether received",
        "total Ether sent",
        "total ether balance"
    ]
].copy()

df_verificacao_saldo["FLAG"] = y_vagifa

df_verificacao_saldo["Saldo_calculado"] = (
    df_verificacao_saldo["total ether received"]
    - df_verificacao_saldo["total Ether sent"]
)

df_verificacao_saldo["Diferenca_absoluta"] = (
    df_verificacao_saldo["total ether balance"]
    - df_verificacao_saldo["Saldo_calculado"]
).abs()

# Registros com diferenças superiores a 0,01 ETH
inconsistencias = df_verificacao_saldo[
    df_verificacao_saldo["Diferenca_absoluta"] > 0.01
]

print("=== REGISTROS COM DIFERENÇAS ===")
display(inconsistencias)

print("\n=== RESUMO DAS DIFERENÇAS ===")

print("Total:", len(inconsistencias))

print(
    "Maior diferença:",
    df_verificacao_saldo["Diferenca_absoluta"].max()
)

print(
    "Diferença mediana:",
    df_verificacao_saldo["Diferenca_absoluta"].median()
)


=== REGISTROS COM DIFERENÇAS ===


,total ether received,total Ether sent,total ether balance,FLAG,Saldo_calculado,Diferenca_absoluta
682,42.180332,45.234944,-3.074612,0,-3.054612,0.020000
6752,5.292232,6.606937,-1.360734,0,-1.314705,0.046029



=== RESUMO DAS DIFERENÇAS ===
Total: 2
Maior diferença: 0.046028712999999666
Diferença mediana: 7.029966886396011e-16



## 10 - Relações entre features e a variável FLAG

Nesta etapa, serão calculadas as correlações de Pearson
e Spearman entre as 22 features numéricas e a
variável-alvo FLAG.

Também serão investigadas relações de alta correlação
entre as próprias features, com o objetivo de identificar
redundâncias.

Os resultados serão utilizados como análise exploratória,
não como prova de causalidade, relevância operacional ou
ausência de target leakage.

Nenhuma feature será removida automaticamente.


In [15]:

import numpy as np
import pandas as pd

# ==========================================
# Correlação das features com FLAG
# ==========================================

df_correlacao_vagifa = X_vagifa.copy()
df_correlacao_vagifa["FLAG"] = y_vagifa

# Correlação de Pearson
corr_pearson = (
    df_correlacao_vagifa
    .corr(method="pearson")["FLAG"]
    .drop("FLAG")
)

# Correlação de Spearman
corr_spearman = (
    df_correlacao_vagifa
    .corr(method="spearman")["FLAG"]
    .drop("FLAG")
)

# Comparar os resultados
df_corr_flag = pd.DataFrame({
    "Feature": X_vagifa.columns,
    "Pearson": corr_pearson.values,
    "Spearman": corr_spearman.values
})

df_corr_flag["Spearman_abs"] = (
    df_corr_flag["Spearman"].abs()
)

df_corr_flag = (
    df_corr_flag
    .sort_values("Spearman_abs", ascending=False)
    .reset_index(drop=True)
)

print("=== CORRELAÇÃO COM FLAG ===")
display(df_corr_flag)

# ==========================================
# Correlação entre as próprias features
# ==========================================

matriz_corr = X_vagifa.corr(method="spearman")

mascara = np.triu(
    np.ones(matriz_corr.shape, dtype=bool),
    k=1
)

pares = (
    matriz_corr
    .where(mascara)
    .stack()
    .reset_index()
)

pares.columns = [
    "Feature_1",
    "Feature_2",
    "Correlacao"
]

pares["Correlacao_abs"] = (
    pares["Correlacao"].abs()
)

pares = pares.sort_values(
    "Correlacao_abs",
    ascending=False
)

print("\n=== PARES COM CORRELAÇÃO ABSOLUTA >= 0.95 ===")

display(
    pares[pares["Correlacao_abs"] >= 0.95]
)


=== CORRELAÇÃO COM FLAG ===


,Feature,Pearson,Spearman,Spearman_abs
0,total ether received,-0.016947,-0.456781,0.456781
1,avg val received,-0.011884,-0.421934,0.421934
2,max value received,-0.019302,-0.406252,0.406252
3,total Ether sent,-0.015034,-0.335721,0.335721
4,Sent tnx,-0.078190,-0.322580,0.322580
5,min value received,-0.021607,-0.294281,0.294281
6,max val sent,-0.022444,-0.290308,0.290308
7,total transactions (including tnx to create co...,-0.100551,-0.263894,0.263894
8,Time Diff between first and last (Mins),-0.269612,-0.261787,0.261787
9,Unique Sent To Addresses,-0.045692,-0.252569,0.252569



=== PARES COM CORRELAÇÃO ABSOLUTA >= 0.95 ===


,Feature_1,Feature_2,Correlacao,Correlacao_abs
210,max val sent to contract,avg value sent to contract,1.000000,1.000000
214,max val sent to contract,total ether sent contracts,1.000000,1.000000
219,avg value sent to contract,total ether sent contracts,1.000000,1.000000
90,Received Tnx,total transactions (including tnx to create co...,0.953874,0.953874
191,max val sent,total Ether sent,0.951454,0.951454
186,max val sent,avg val sent,0.951065,0.951065



## 11 - Investigação de features altamente correlacionadas

A análise de Spearman identificou correlação igual a 1 entre
três atributos relacionados ao envio de Ether para contratos.

Essas variáveis também apresentam frequência de zeros
superior a 99,9%.

Será investigado se a correlação perfeita resulta de
valores efetivamente equivalentes ou da presença
predominante de zeros nas mesmas observações.

Também será analisada a distribuição dos registros
não zerados entre as classes FLAG = 0 e FLAG = 1.

Nenhuma feature será removida antes dessa verificação.


In [16]:

# ==========================================
# Auditoria das features de contratos
# ==========================================

colunas_contratos = [
    "max val sent to contract",
    "avg value sent to contract",
    "total ether sent contracts"
]

df_contratos = X_vagifa[colunas_contratos].copy()
df_contratos["FLAG"] = y_vagifa

print("=== VALORES DIFERENTES DE ZERO ===")

resumo_contratos = pd.DataFrame({
    "Valores_unicos": X_vagifa[colunas_contratos].nunique(),
    "Registros_nao_zero": (
        X_vagifa[colunas_contratos] != 0
    ).sum(),
    "Percentual_zero": (
        (X_vagifa[colunas_contratos] == 0).mean() * 100
    ).round(4)
})

display(resumo_contratos)

print("\n=== REGISTROS COM VALORES NÃO ZERADOS ===")

tem_movimentacao = (
    X_vagifa[colunas_contratos] != 0
).any(axis=1)

print("Quantidade:", tem_movimentacao.sum())

display(df_contratos.loc[tem_movimentacao])

print("\n=== DISTRIBUIÇÃO POR CLASSE ===")

display(
    pd.crosstab(
        y_vagifa,
        tem_movimentacao,
        rownames=["FLAG"],
        colnames=["Algum valor diferente de zero"]
    )
)

print("\n=== COMPARAÇÃO ENTRE AS COLUNAS ===")

for i, coluna_a in enumerate(colunas_contratos):
    for coluna_b in colunas_contratos[i+1:]:

        iguais = (
            X_vagifa[coluna_a] == X_vagifa[coluna_b]
        ).sum()

        print(
            f"{coluna_a} x {coluna_b}: "
            f"{iguais} valores iguais em {len(X_vagifa)} registros"
        )


=== VALORES DIFERENTES DE ZERO ===


,Valores_unicos,Registros_nao_zero,Percentual_zero
max val sent to contract,4,3,99.9694
avg value sent to contract,4,3,99.9694
total ether sent contracts,4,3,99.9694



=== REGISTROS COM VALORES NÃO ZERADOS ===
Quantidade: 3


,max val sent to contract,avg value sent to contract,total ether sent contracts,FLAG
682,0.020000,0.020000,0.020000,0
5738,0.010000,0.010000,0.010000,0
6752,0.046029,0.023014,0.046029,0



=== DISTRIBUIÇÃO POR CLASSE ===


Algum valor diferente de zero,False,True
FLAG,,
0,7634,3
1,2179,0



=== COMPARAÇÃO ENTRE AS COLUNAS ===
max val sent to contract x avg value sent to contract: 9815 valores iguais em 9816 registros
max val sent to contract x total ether sent contracts: 9815 valores iguais em 9816 registros
avg value sent to contract x total ether sent contracts: 9815 valores iguais em 9816 registros



## 12 - Verificação da fórmula de total ether balance

A análise das features relacionadas a contratos identificou
apenas três registros com valores diferentes de zero.

Duas diferenças encontradas anteriormente no cálculo do saldo
coincidem com os valores de Ether enviados para contratos.

Será testada a hipótese de que `total ether balance` considera
separadamente o Ether enviado em transações comuns e o Ether
enviado para contratos.

A fórmula será verificada em todos os 9.816 registros,
sem alterar os valores originais.


In [17]:

import numpy as np
import pandas as pd

# ==========================================
# Verificação das fórmulas do saldo
# ==========================================

recebido = X_vagifa["total ether received"]
enviado = X_vagifa["total Ether sent"]
contratos = X_vagifa["total ether sent contracts"]
saldo_original = X_vagifa["total ether balance"]

# Fórmula anterior
saldo_formula_1 = recebido - enviado

# Nova hipótese: considerar envios para contratos
saldo_formula_2 = recebido - enviado - contratos

erro_1 = (saldo_original - saldo_formula_1).abs()
erro_2 = (saldo_original - saldo_formula_2).abs()

print("=== FÓRMULA 1: RECEBIDO - ENVIADO ===")
print("Diferenças > 0,000001 ETH:", (erro_1 > 1e-6).sum())
print("Maior diferença:", erro_1.max())

print("\n=== FÓRMULA 2: RECEBIDO - ENVIADO - CONTRATOS ===")
print("Diferenças > 0,000001 ETH:", (erro_2 > 1e-6).sum())
print("Maior diferença:", erro_2.max())
print("Diferença mediana:", erro_2.median())

print("\n=== OS TRÊS REGISTROS COM ENVIO PARA CONTRATOS ===")

verificacao = pd.DataFrame({
    "FLAG": y_vagifa,
    "Saldo_original": saldo_original,
    "Saldo_formula_1": saldo_formula_1,
    "Saldo_formula_2": saldo_formula_2,
    "Ether_contratos": contratos,
    "Erro_formula_1": erro_1,
    "Erro_formula_2": erro_2
})

display(verificacao.loc[contratos != 0])


=== FÓRMULA 1: RECEBIDO - ENVIADO ===
Diferenças > 0,000001 ETH: 131
Maior diferença: 0.046028712999999666

=== FÓRMULA 2: RECEBIDO - ENVIADO - CONTRATOS ===
Diferenças > 0,000001 ETH: 129
Maior diferença: 0.0017338997017759539
Diferença mediana: 7.029966886396011e-16

=== OS TRÊS REGISTROS COM ENVIO PARA CONTRATOS ===


,FLAG,Saldo_original,Saldo_formula_1,Saldo_formula_2,Ether_contratos,Erro_formula_1,Erro_formula_2
682,0,-3.074612,-3.054612,-3.074612,0.020000,0.020000,4.440892e-16
5738,0,2108.645197,2108.655190,2108.645190,0.010000,0.009993,6.999999e-06
6752,0,-1.360734,-1.314705,-1.360734,0.046029,0.046029,4.440892e-16


In [18]:

# ==========================================
# Análise das diferenças residuais do saldo
# ==========================================

erro_residual = (
    saldo_original - saldo_formula_2
).abs()

limiares = [
    1e-8,
    1e-6,
    1e-5,
    1e-4,
    1e-3,
    1e-2
]

print("=== DIFERENÇAS POR TOLERÂNCIA ===")

for limite in limiares:
    quantidade = (erro_residual > limite).sum()
    print(f"Erro > {limite:.8f} ETH: {quantidade}")

print("\n=== DEZ MAIORES DIFERENÇAS ===")

df_erros = pd.DataFrame({
    "FLAG": y_vagifa,
    "Saldo_original": saldo_original,
    "Saldo_calculado": saldo_formula_2,
    "Erro_absoluto": erro_residual
})

display(
    df_erros.sort_values(
        "Erro_absoluto",
        ascending=False
    ).head(10)
)


=== DIFERENÇAS POR TOLERÂNCIA ===
Erro > 0.00000001 ETH: 2323
Erro > 0.00000100 ETH: 129
Erro > 0.00001000 ETH: 37
Erro > 0.00010000 ETH: 8
Erro > 0.00100000 ETH: 2
Erro > 0.01000000 ETH: 0

=== DEZ MAIORES DIFERENÇAS ===


,FLAG,Saldo_original,Saldo_calculado,Erro_absoluto
145,0,6.291783e+02,6.291800e+02,0.001734
1503,0,-3.395546e+06,-3.395546e+06,0.001000
4797,0,-5.408058e+05,-5.408058e+05,0.000600
525,0,2.084272e+03,2.084272e+03,0.000476
5163,0,1.076460e+00,1.076000e+00,0.000460
2640,0,-5.484555e+03,-5.484555e+03,0.000448
5781,0,2.868641e+02,2.868640e+02,0.000140
158,0,-7.525769e+05,-7.525769e+05,0.000100
914,0,-4.718786e+05,-4.718786e+05,0.000100
1259,0,1.014480e+03,1.014480e+03,0.000073



## 13 - Documentação das features do dataset Vagifa

Após a auditoria de qualidade, será construída uma tabela
consolidada das 22 features numéricas utilizadas no primeiro
cenário experimental.

A tabela incluirá tipos de dados, quantidade de valores
distintos, medianas por classe, frequência de zeros e
correlação de Spearman com FLAG.

O objetivo é documentar as características das variáveis
e subsidiar a análise de relevância, redundância e possíveis
proxies da variável-alvo.

A interpretação estatística não substitui a investigação
da origem e do significado de cada atributo.


In [19]:

# ==========================================
# Dicionário estatístico das 22 features
# ==========================================

corr_flag = (
    pd.concat(
        [X_vagifa, y_vagifa.rename("FLAG")],
        axis=1
    )
    .corr(method="spearman")["FLAG"]
    .drop("FLAG")
)

df_dicionario = pd.DataFrame({
    "Feature": X_vagifa.columns,
    "Tipo": X_vagifa.dtypes.astype(str).values,
    "Valores_unicos": X_vagifa.nunique().values,

    "Mediana_FLAG_0": (
        X_vagifa.loc[y_vagifa == 0]
        .median().values
    ),

    "Mediana_FLAG_1": (
        X_vagifa.loc[y_vagifa == 1]
        .median().values
    ),

    "Zeros_FLAG_0_%": (
        (X_vagifa.loc[y_vagifa == 0] == 0)
        .mean().mul(100).values
    ),

    "Zeros_FLAG_1_%": (
        (X_vagifa.loc[y_vagifa == 1] == 0)
        .mean().mul(100).values
    ),

    "Spearman_FLAG": corr_flag.values
})

df_dicionario["Correlacao_absoluta"] = (
    df_dicionario["Spearman_FLAG"].abs()
)

df_dicionario = (
    df_dicionario
    .sort_values(
        "Correlacao_absoluta",
        ascending=False
    )
    .reset_index(drop=True)
)

display(df_dicionario.round(4))

print("Total de features:", len(df_dicionario))


,Feature,Tipo,Valores_unicos,Mediana_FLAG_0,Mediana_FLAG_1,Zeros_FLAG_0_%,Zeros_FLAG_1_%,Spearman_FLAG,Correlacao_absoluta
0,total ether received,float64,6728,71.6674,1.6788,0.6547,30.1514,-0.4568,0.4568
1,avg val received,float64,6767,3.7500,0.5001,0.6809,30.1514,-0.4219,0.4219
2,max value received,float64,6302,12.2283,0.9979,0.6547,30.1514,-0.4063,0.4063
3,total Ether sent,float64,5868,34.0828,0.9991,16.1320,38.0909,-0.3357,0.3357
4,Sent tnx,int64,641,3.0000,1.0000,15.5689,37.6778,-0.3226,0.3226
5,min value received,float64,4589,0.1562,0.0045,16.4201,37.3566,-0.2943,0.2943
6,max val sent,float64,6647,9.7000,0.6880,16.1320,38.0909,-0.2903,0.2903
7,total transactions (including tnx to create co...,int64,897,11.0000,5.0000,0.0000,24.2772,-0.2639,0.2639
8,Time Diff between first and last (Mins),float64,7810,116874.3200,7545.4300,0.6023,26.7554,-0.2618,0.2618
9,Unique Sent To Addresses,int64,258,2.0000,1.0000,15.5689,37.6778,-0.2526,0.2526


Total de features: 22



## 14 - Dicionário descritivo das features

As 22 features numéricas do Vagifa serão organizadas
por grupos de características: temporais, transacionais,
relacionamentos, valores recebidos, valores enviados,
contratos e saldo.

Cada atributo receberá uma descrição preliminar,
acompanhada das estatísticas calculadas anteriormente.

As descrições serão posteriormente verificadas com a
documentação da origem do dataset.

O objetivo é preparar a documentação exigida pela
pré-banca e identificar atributos redundantes ou
potencialmente relacionados à construção do rótulo.


In [20]:

# ==========================================
# Dicionário descritivo - Dataset Vagifa
# ==========================================

# Localizar a coluna de total de transações
col_total_transacoes = next(
    c for c in X_vagifa.columns
    if c.lower().startswith("total transactions")
)

# Grupo e significado preliminar
mapa_features = {
    "Avg min between sent tnx":
        ("Temporal", "Intervalo médio entre transações enviadas, em minutos"),

    "Avg min between received tnx":
        ("Temporal", "Intervalo médio entre transações recebidas, em minutos"),

    "Time Diff between first and last (Mins)":
        ("Temporal", "Tempo entre primeira e última transação, em minutos"),

    "Sent tnx":
        ("Atividade", "Quantidade de transações enviadas"),

    "Received Tnx":
        ("Atividade", "Quantidade de transações recebidas"),

    col_total_transacoes:
        ("Atividade", "Quantidade total de transações, incluindo criação de contratos"),

    "Number of Created Contracts":
        ("Atividade", "Quantidade de contratos criados"),

    "Unique Received From Addresses":
        ("Relacionamentos", "Quantidade de endereços distintos de origem"),

    "Unique Sent To Addresses":
        ("Relacionamentos", "Quantidade de endereços distintos de destino"),

    "min value received":
        ("Recebimentos", "Menor valor recebido"),

    "max value received":
        ("Recebimentos", "Maior valor recebido"),

    "avg val received":
        ("Recebimentos", "Valor médio recebido"),

    "total ether received":
        ("Recebimentos", "Total de Ether recebido"),

    "min val sent":
        ("Envios", "Menor valor enviado"),

    "max val sent":
        ("Envios", "Maior valor enviado"),

    "avg val sent":
        ("Envios", "Valor médio enviado"),

    "total Ether sent":
        ("Envios", "Total de Ether enviado"),

    "min value sent to contract":
        ("Contratos", "Menor valor enviado para contrato"),

    "max val sent to contract":
        ("Contratos", "Maior valor enviado para contrato"),

    "avg value sent to contract":
        ("Contratos", "Valor médio enviado para contratos"),

    "total ether sent contracts":
        ("Contratos", "Total de Ether enviado para contratos"),

    "total ether balance":
        ("Saldo", "Saldo líquido calculado, aproximadamente")
}

# Criar tabela com as descrições
df_glossario_vagifa = df_dicionario.copy()

df_glossario_vagifa["Grupo"] = (
    df_glossario_vagifa["Feature"]
    .map(lambda x: mapa_features.get(x, ("Não classificado", ""))[0])
)

df_glossario_vagifa["Descricao_preliminar"] = (
    df_glossario_vagifa["Feature"]
    .map(lambda x: mapa_features.get(x, ("", "Não definida"))[1])
)

# Verificar se todas as features foram classificadas
nao_classificadas = df_glossario_vagifa[
    df_glossario_vagifa["Grupo"] == "Não classificado"
]

print("Total de features:", len(df_glossario_vagifa))
print("Features não classificadas:", len(nao_classificadas))

if len(nao_classificadas) > 0:
    print("\nVerificar nomes destas colunas:")
    display(nao_classificadas[["Feature"]])

print("\n=== DICIONÁRIO DE FEATURES ===")

display(
    df_glossario_vagifa[
        [
            "Feature",
            "Grupo",
            "Descricao_preliminar",
            "Spearman_FLAG"
        ]
    ]
)


Total de features: 22
Features não classificadas: 0

=== DICIONÁRIO DE FEATURES ===


,Feature,Grupo,Descricao_preliminar,Spearman_FLAG
0,total ether received,Recebimentos,Total de Ether recebido,-0.456781
1,avg val received,Recebimentos,Valor médio recebido,-0.421934
2,max value received,Recebimentos,Maior valor recebido,-0.406252
3,total Ether sent,Envios,Total de Ether enviado,-0.335721
4,Sent tnx,Atividade,Quantidade de transações enviadas,-0.322580
5,min value received,Recebimentos,Menor valor recebido,-0.294281
6,max val sent,Envios,Maior valor enviado,-0.290308
7,total transactions (including tnx to create co...,Atividade,"Quantidade total de transações, incluindo cria...",-0.263894
8,Time Diff between first and last (Mins),Temporal,"Tempo entre primeira e última transação, em mi...",-0.261787
9,Unique Sent To Addresses,Relacionamentos,Quantidade de endereços distintos de destino,-0.252569



## 15 - Divisão dos dados para os experimentos

Após a auditoria e a preparação das 22 features,
será realizada uma divisão estratificada dos dados
em treinamento (60%), validação (20%) e teste (20%).

O conjunto de treinamento será utilizado para ajustar
os modelos e as transformações.

O conjunto de validação será utilizado para selecionar
configurações, atributos e thresholds.

O conjunto de teste será reservado para avaliação final,
evitando sua utilização durante as decisões de modelagem.

Como a base foi deduplicada por endereço Ethereum,
cada conta aparece apenas uma vez nos conjuntos preparados.

O particionamento é aleatório estratificado, não temporal.
Essa limitação será considerada na interpretação dos resultados.


In [21]:

from sklearn.model_selection import train_test_split

# Dados preparados: 22 features, sem ERC-20
X = X_vagifa.copy()
y = y_vagifa.copy()

# Separar 20% para teste final
X_temp, X_test_vagifa, y_temp, y_test_vagifa = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Do restante, separar 25% para validação
# Resultado: 60% treino, 20% validação, 20% teste
X_train_vagifa, X_val_vagifa, y_train_vagifa, y_val_vagifa = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    stratify=y_temp,
    random_state=42
)

print("=== DIMENSÕES ===")
print("Treino:", X_train_vagifa.shape)
print("Validação:", X_val_vagifa.shape)
print("Teste:", X_test_vagifa.shape)

print("\n=== DISTRIBUIÇÃO DAS CLASSES ===")

for nome, alvo in [
    ("Treinamento", y_train_vagifa),
    ("Validação", y_val_vagifa),
    ("Teste", y_test_vagifa)
]:
    print(f"\n{nome}")
    print(alvo.value_counts())
    print((alvo.value_counts(normalize=True) * 100).round(2))


=== DIMENSÕES ===
Treino: (5889, 22)
Validação: (1963, 22)
Teste: (1964, 22)

=== DISTRIBUIÇÃO DAS CLASSES ===

Treinamento
FLAG
0    4582
1    1307
Name: count, dtype: int64
FLAG
0    77.81
1    22.19
Name: proportion, dtype: float64

Validação
FLAG
0    1527
1     436
Name: count, dtype: int64
FLAG
0    77.79
1    22.21
Name: proportion, dtype: float64

Teste
FLAG
0    1528
1     436
Name: count, dtype: int64
FLAG
0    77.8
1    22.2
Name: proportion, dtype: float64



## 16 - Definição dos cenários experimentais do Vagifa

Após a limpeza e a divisão estratificada em treinamento,
validação e teste, foram definidos quatro cenários de features.

O Cenário A utiliza todas as 22 features numéricas preparadas.

O Cenário B remove a variável `total ether balance`, cuja
relação matemática com os valores recebidos e enviados
foi identificada durante a auditoria.

O Cenário C remove quatro atributos relacionados a envios
para contratos, que apresentam mais de 99,9% de zeros.

O Cenário D utiliza somente atributos de atividade e
relacionamentos entre contas Ethereum.

Os cenários serão comparados usando os mesmos registros
de treinamento e validação.

O conjunto de teste permanecerá reservado para avaliação
posterior, após a definição das configurações dos modelos.


In [22]:

# ==========================================
# Cenários de features - Vagifa
# ==========================================

features_A_vagifa = X_train_vagifa.columns.tolist()

# Feature aproximadamente derivada de outras variáveis
features_B_vagifa = [
    c for c in features_A_vagifa
    if c != "total ether balance"
]

# Features extremamente esparsas de contratos
features_contratos = [
    "min value sent to contract",
    "max val sent to contract",
    "avg value sent to contract",
    "total ether sent contracts"
]

features_C_vagifa = [
    c for c in features_A_vagifa
    if c not in features_contratos
]

# Identificar a coluna de total de transações
col_total_transacoes = next(
    c for c in features_A_vagifa
    if c.lower().startswith("total transactions")
)

# Apenas atividade e relacionamentos
features_D_vagifa = [
    "Sent tnx",
    "Received Tnx",
    col_total_transacoes,
    "Number of Created Contracts",
    "Unique Received From Addresses",
    "Unique Sent To Addresses"
]

cenarios_vagifa = {
    "A_Completo": features_A_vagifa,
    "B_Sem_Saldo": features_B_vagifa,
    "C_Sem_Contratos": features_C_vagifa,
    "D_Atividade_Relacionamentos": features_D_vagifa
}

# Validar e exibir os cenários
for nome, features in cenarios_vagifa.items():

    faltantes = set(features) - set(X_train_vagifa.columns)
    assert not faltantes, f"Features ausentes: {faltantes}"

    print(f"\n=== {nome} ===")
    print("Quantidade de features:", len(features))
    print("Treino:", X_train_vagifa[features].shape)
    print("Validação:", X_val_vagifa[features].shape)
    print("Features:", features)



=== A_Completo ===
Quantidade de features: 22
Treino: (5889, 22)
Validação: (1963, 22)
Features: ['Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 'min value received', 'max value received', 'avg val received', 'min val sent', 'max val sent', 'avg val sent', 'min value sent to contract', 'max val sent to contract', 'avg value sent to contract', 'total transactions (including tnx to create contract', 'total Ether sent', 'total ether received', 'total ether sent contracts', 'total ether balance']

=== B_Sem_Saldo ===
Quantidade de features: 21
Treino: (5889, 21)
Validação: (1963, 21)
Features: ['Avg min between sent tnx', 'Avg min between received tnx', 'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx', 'Number of Created Contracts', 'Unique Received From Addresses', 'Unique Sent To Addresses', 


## 17 - Regressão Logística nos quatro cenários

A Regressão Logística será utilizada como primeiro modelo
de referência para o dataset Vagifa.

Serão avaliados os quatro cenários de features previamente
definidos, utilizando os mesmos conjuntos de treinamento
e validação.

O pré-processamento será realizado com StandardScaler
dentro de um Pipeline, garantindo que os parâmetros de
padronização sejam ajustados somente no treinamento.

As avaliações utilizarão Precision, Recall, F1-Score,
PR-AUC trapezoidal e Average Precision (AP).

A acurácia será apresentada apenas como métrica
complementar.

Inicialmente será utilizado o threshold padrão de 0,5,
sem ajuste de hiperparâmetros ou balanceamento artificial.

O conjunto de teste permanecerá reservado.


In [23]:

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    confusion_matrix
)

# ==========================================
# Regressão Logística - Dataset Vagifa
# ==========================================

resultados_logistica_vagifa = []
modelos_logistica_vagifa = {}

for nome_cenario, features in cenarios_vagifa.items():

    X_treino = X_train_vagifa[features]
    X_validacao = X_val_vagifa[features]

    # Pipeline com padronização e modelo
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=3000,
            random_state=42
        ))
    ])

    # Ajustar somente no treinamento
    pipeline.fit(X_treino, y_train_vagifa)

    # Predições na validação
    y_pred = pipeline.predict(X_validacao)
    y_prob = pipeline.predict_proba(X_validacao)[:, 1]

    # Área trapezoidal Precision-Recall
    precision_curva, recall_curva, _ = (
        precision_recall_curve(y_val_vagifa, y_prob)
    )

    pr_auc = auc(recall_curva, precision_curva)

    # Average Precision
    ap = average_precision_score(
        y_val_vagifa,
        y_prob
    )

    # Matriz de confusão
    matriz = confusion_matrix(
        y_val_vagifa,
        y_pred,
        labels=[0, 1]
    )

    resultados_logistica_vagifa.append({
        "Cenario": nome_cenario,
        "Features": len(features),
        "Precision": precision_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "AP": ap,
        "PR_AUC": pr_auc,
        "Accuracy": accuracy_score(
            y_val_vagifa, y_pred
        )
    })

    modelos_logistica_vagifa[nome_cenario] = pipeline

    print(f"\n=== {nome_cenario} ===")
    print("Matriz de confusão:")
    print(matriz)

# Tabela comparativa
df_resultados_logistica_vagifa = pd.DataFrame(
    resultados_logistica_vagifa
)

print("\n=== COMPARAÇÃO DOS QUATRO CENÁRIOS ===")

display(
    df_resultados_logistica_vagifa.round(4)
)

print(
    "\nProporção de fraude na validação:",
    round(y_val_vagifa.mean(), 4)
)



=== A_Completo ===
Matriz de confusão:
[[1526    1]
 [ 436    0]]

=== B_Sem_Saldo ===
Matriz de confusão:
[[1527    0]
 [ 436    0]]

=== C_Sem_Contratos ===
Matriz de confusão:
[[1527    0]
 [ 436    0]]

=== D_Atividade_Relacionamentos ===
Matriz de confusão:
[[1527    0]
 [ 436    0]]

=== COMPARAÇÃO DOS QUATRO CENÁRIOS ===


,Cenario,Features,Precision,Recall,F1,AP,PR_AUC,Accuracy
0,A_Completo,22,0.0,0.0,0.0,0.4788,0.4273,0.7774
1,B_Sem_Saldo,21,0.0,0.0,0.0,0.4774,0.4260,0.7779
2,C_Sem_Contratos,18,0.0,0.0,0.0,0.4843,0.4329,0.7779
3,D_Atividade_Relacionamentos,6,0.0,0.0,0.0,0.6099,0.5758,0.7779



Proporção de fraude na validação: 0.2221



## 18 - Investigação das probabilidades da Regressão Logística

Na primeira comparação dos quatro cenários, a Regressão
Logística praticamente não identificou registros fraudulentos
utilizando o limiar padrão de classificação de 0,5.

Apesar disso, os resultados de Average Precision indicaram
capacidade discriminativa superior à referência baseada na
prevalência da classe fraudulenta.

Será investigada a distribuição das probabilidades
produzidas pelos modelos e o comportamento da classificação
sob diferentes limiares.

A investigação utilizará exclusivamente o conjunto de
validação, mantendo o teste final reservado.


In [24]:

import numpy as np
import pandas as pd

# ==========================================
# Distribuição das probabilidades
# ==========================================

resultados_probabilidades = []
resultados_thresholds = []

for nome, modelo in modelos_logistica_vagifa.items():

    features = cenarios_vagifa[nome]

    probabilidades = modelo.predict_proba(
        X_val_vagifa[features]
    )[:, 1]

    classe_real = y_val_vagifa.to_numpy()

    probabilidades_normais = probabilidades[
        classe_real == 0
    ]

    probabilidades_fraudes = probabilidades[
        classe_real == 1
    ]

    resultados_probabilidades.append({
        "Cenario": nome,
        "Mediana_FLAG_0": np.median(probabilidades_normais),
        "Mediana_FLAG_1": np.median(probabilidades_fraudes),
        "P95_FLAG_1": np.percentile(probabilidades_fraudes, 95),
        "Probabilidade_maxima": probabilidades.max(),
        "Predicoes_fraude_0.5": int(
            (probabilidades >= 0.5).sum()
        )
    })

    # Avaliação exploratória de diferentes thresholds
    for threshold in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:

        predicao = (
            probabilidades >= threshold
        ).astype(int)

        resultados_thresholds.append({
            "Cenario": nome,
            "Threshold": threshold,
            "Preditos_fraude": int(predicao.sum()),
            "Precision": precision_score(
                classe_real, predicao, zero_division=0
            ),
            "Recall": recall_score(
                classe_real, predicao, zero_division=0
            ),
            "F1": f1_score(
                classe_real, predicao, zero_division=0
            )
        })

print("=== DISTRIBUIÇÃO DAS PROBABILIDADES ===")

display(
    pd.DataFrame(resultados_probabilidades).round(4)
)

print("\n=== COMPARAÇÃO DE THRESHOLDS ===")

df_thresholds = pd.DataFrame(resultados_thresholds)

display(df_thresholds.round(4))


=== DISTRIBUIÇÃO DAS PROBABILIDADES ===


,Cenario,Mediana_FLAG_0,Mediana_FLAG_1,P95_FLAG_1,Probabilidade_maxima,Predicoes_fraude_0.5
0,A_Completo,0.1646,0.3692,0.3791,0.9942,1
1,B_Sem_Saldo,0.1643,0.3692,0.3791,0.3801,0
2,C_Sem_Contratos,0.1650,0.3694,0.3792,0.3799,0
3,D_Atividade_Relacionamentos,0.2645,0.2748,0.2783,0.2791,0



=== COMPARAÇÃO DE THRESHOLDS ===


,Cenario,Threshold,Preditos_fraude,Precision,Recall,F1
0,A_Completo,0.05,1536,0.2760,0.9725,0.4300
1,A_Completo,0.10,1386,0.3038,0.9656,0.4621
2,A_Completo,0.20,1088,0.3483,0.8693,0.4974
3,A_Completo,0.30,943,0.3743,0.8096,0.5120
4,A_Completo,0.40,1,0.0000,0.0000,0.0000
5,A_Completo,0.50,1,0.0000,0.0000,0.0000
6,B_Sem_Saldo,0.05,1534,0.2764,0.9725,0.4305
7,B_Sem_Saldo,0.10,1384,0.3042,0.9656,0.4626
8,B_Sem_Saldo,0.20,1084,0.3496,0.8693,0.4987
9,B_Sem_Saldo,0.30,940,0.3755,0.8096,0.5131



## 19 - Regressão Logística com pesos de classe

A análise das probabilidades demonstrou que a Regressão
Logística produziu probabilidades predominantemente
inferiores ao threshold padrão de 0,5.

A redução do limiar permitiu aumentar o Recall, mas
a Precision permaneceu relativamente baixa.

Nesta etapa, será investigado o uso de
`class_weight="balanced"`, que atribui pesos inversamente
proporcionais à frequência das classes durante o treinamento.

O objetivo é verificar se o tratamento do desbalanceamento
melhora a identificação dos registros fraudulentos.

Serão mantidos o mesmo pré-processamento, os quatro cenários
de features e os conjuntos de treinamento e validação.

O conjunto de teste permanecerá reservado.


In [25]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    confusion_matrix
)
import pandas as pd

# ==========================================
# Regressão Logística com class_weight
# ==========================================

resultados_logistica_balanceada = []
modelos_logistica_balanceada = {}

for nome, features in cenarios_vagifa.items():

    X_treino = X_train_vagifa[features]
    X_validacao = X_val_vagifa[features]

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    pipeline.fit(X_treino, y_train_vagifa)

    y_pred = pipeline.predict(X_validacao)
    y_prob = pipeline.predict_proba(X_validacao)[:, 1]

    precision_curva, recall_curva, _ = precision_recall_curve(
        y_val_vagifa, y_prob
    )

    resultados_logistica_balanceada.append({
        "Cenario": nome,
        "Precision": precision_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "AP": average_precision_score(
            y_val_vagifa, y_prob
        ),
        "PR_AUC": auc(recall_curva, precision_curva),
        "Accuracy": accuracy_score(
            y_val_vagifa, y_pred
        )
    })

    modelos_logistica_balanceada[nome] = pipeline

    print(f"\n=== {nome} ===")
    print("Matriz de confusão:")
    print(confusion_matrix(
        y_val_vagifa, y_pred, labels=[0, 1]
    ))

df_resultados_logistica_balanceada = pd.DataFrame(
    resultados_logistica_balanceada
)

print("\n=== RESULTADOS COM PESOS DE CLASSE ===")
display(df_resultados_logistica_balanceada.round(4))

print("\n=== COMPARAÇÃO COM MODELO ORIGINAL ===")

comparacao_pesos = pd.concat([
    df_resultados_logistica_vagifa.assign(
        Configuracao="Original"
    ),
    df_resultados_logistica_balanceada.assign(
        Configuracao="Balanceada"
    )
])

display(
    comparacao_pesos[
        ["Configuracao", "Cenario",
         "Precision", "Recall", "F1", "AP", "PR_AUC"]
    ].round(4)
)



=== A_Completo ===
Matriz de confusão:
[[856 671]
 [ 65 371]]

=== B_Sem_Saldo ===
Matriz de confusão:
[[855 672]
 [ 65 371]]

=== C_Sem_Contratos ===
Matriz de confusão:
[[856 671]
 [ 65 371]]

=== D_Atividade_Relacionamentos ===
Matriz de confusão:
[[ 497 1030]
 [  25  411]]

=== RESULTADOS COM PESOS DE CLASSE ===


,Cenario,Precision,Recall,F1,AP,PR_AUC,Accuracy
0,A_Completo,0.3560,0.8509,0.5020,0.4849,0.4417,0.6251
1,B_Sem_Saldo,0.3557,0.8509,0.5017,0.4865,0.4426,0.6246
2,C_Sem_Contratos,0.3560,0.8509,0.5020,0.4891,0.4451,0.6251
3,D_Atividade_Relacionamentos,0.2852,0.9427,0.4379,0.6094,0.5723,0.4626



=== COMPARAÇÃO COM MODELO ORIGINAL ===


,Configuracao,Cenario,Precision,Recall,F1,AP,PR_AUC
0,Original,A_Completo,0.0000,0.0000,0.0000,0.4788,0.4273
1,Original,B_Sem_Saldo,0.0000,0.0000,0.0000,0.4774,0.4260
2,Original,C_Sem_Contratos,0.0000,0.0000,0.0000,0.4843,0.4329
3,Original,D_Atividade_Relacionamentos,0.0000,0.0000,0.0000,0.6099,0.5758
0,Balanceada,A_Completo,0.3560,0.8509,0.5020,0.4849,0.4417
1,Balanceada,B_Sem_Saldo,0.3557,0.8509,0.5017,0.4865,0.4426
2,Balanceada,C_Sem_Contratos,0.3560,0.8509,0.5020,0.4891,0.4451
3,Balanceada,D_Atividade_Relacionamentos,0.2852,0.9427,0.4379,0.6094,0.5723



## 20 - Avaliação inicial do XGBoost

O XGBoost será utilizado como segundo algoritmo supervisionado
nos experimentos com o dataset Vagifa.

Serão avaliados os quatro cenários de features definidos
anteriormente, mantendo os mesmos conjuntos de treinamento
e validação.

Inicialmente, será utilizado um conjunto fixo de
hiperparâmetros, sem aplicação de balanceamento artificial
e sem ajuste de threshold.

A Average Precision (AP) será utilizada como métrica
principal, acompanhada de Precision, Recall, F1 e PR-AUC
trapezoidal.

O conjunto de teste permanecerá reservado.


In [26]:

from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    confusion_matrix
)
import pandas as pd

# ==========================================
# XGBoost - Quatro cenários do Vagifa
# ==========================================

resultados_xgb_vagifa = []
modelos_xgb_vagifa = {}

for nome, features in cenarios_vagifa.items():

    X_treino = X_train_vagifa[features]
    X_validacao = X_val_vagifa[features]

    modelo = XGBClassifier(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
        tree_method="hist"
    )

    # Treinamento
    modelo.fit(X_treino, y_train_vagifa)

    # Predições na validação
    y_pred = modelo.predict(X_validacao)
    y_prob = modelo.predict_proba(X_validacao)[:, 1]

    # Curva Precision-Recall
    precision_curva, recall_curva, _ = (
        precision_recall_curve(y_val_vagifa, y_prob)
    )

    # Métricas
    resultados_xgb_vagifa.append({
        "Cenario": nome,
        "Features": len(features),
        "Precision": precision_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_val_vagifa, y_pred, zero_division=0
        ),
        "AP": average_precision_score(
            y_val_vagifa, y_prob
        ),
        "PR_AUC": auc(
            recall_curva, precision_curva
        )
    })

    modelos_xgb_vagifa[nome] = modelo

    print(f"\n=== {nome} ===")
    print("Matriz de confusão:")
    print(confusion_matrix(
        y_val_vagifa, y_pred, labels=[0, 1]
    ))

df_resultados_xgb_vagifa = pd.DataFrame(
    resultados_xgb_vagifa
)

print("\n=== RESULTADOS DO XGBOOST ===")
display(df_resultados_xgb_vagifa.round(4))

print("\n=== COMPARAÇÃO COM REGRESSÃO LOGÍSTICA ===")

comparacao_vagifa = pd.concat([
    df_resultados_logistica_vagifa.assign(
        Modelo="Logística original"
    ),
    df_resultados_logistica_balanceada.assign(
        Modelo="Logística balanceada"
    ),
    df_resultados_xgb_vagifa.assign(
        Modelo="XGBoost"
    )
])

display(
    comparacao_vagifa[
        ["Modelo", "Cenario", "Precision",
         "Recall", "F1", "AP", "PR_AUC"]
    ].round(4)
)



=== A_Completo ===
Matriz de confusão:
[[1494   33]
 [  91  345]]

=== B_Sem_Saldo ===
Matriz de confusão:
[[1494   33]
 [  93  343]]

=== C_Sem_Contratos ===
Matriz de confusão:
[[1493   34]
 [  88  348]]

=== D_Atividade_Relacionamentos ===
Matriz de confusão:
[[1504   23]
 [ 169  267]]

=== RESULTADOS DO XGBOOST ===


,Cenario,Features,Precision,Recall,F1,AP,PR_AUC
0,A_Completo,22,0.9127,0.7913,0.8477,0.9430,0.9429
1,B_Sem_Saldo,21,0.9122,0.7867,0.8448,0.9396,0.9396
2,C_Sem_Contratos,18,0.9110,0.7982,0.8509,0.9430,0.9429
3,D_Atividade_Relacionamentos,6,0.9207,0.6124,0.7355,0.8458,0.8517



=== COMPARAÇÃO COM REGRESSÃO LOGÍSTICA ===


,Modelo,Cenario,Precision,Recall,F1,AP,PR_AUC
0,Logística original,A_Completo,0.0000,0.0000,0.0000,0.4788,0.4273
1,Logística original,B_Sem_Saldo,0.0000,0.0000,0.0000,0.4774,0.4260
2,Logística original,C_Sem_Contratos,0.0000,0.0000,0.0000,0.4843,0.4329
3,Logística original,D_Atividade_Relacionamentos,0.0000,0.0000,0.0000,0.6099,0.5758
0,Logística balanceada,A_Completo,0.3560,0.8509,0.5020,0.4849,0.4417
1,Logística balanceada,B_Sem_Saldo,0.3557,0.8509,0.5017,0.4865,0.4426
2,Logística balanceada,C_Sem_Contratos,0.3560,0.8509,0.5020,0.4891,0.4451
3,Logística balanceada,D_Atividade_Relacionamentos,0.2852,0.9427,0.4379,0.6094,0.5723
0,XGBoost,A_Completo,0.9127,0.7913,0.8477,0.9430,0.9429
1,XGBoost,B_Sem_Saldo,0.9122,0.7867,0.8448,0.9396,0.9396



## 21 - Auditoria de perfis repetidos entre treino e validação

Embora tenham sido removidos os endereços Ethereum
duplicados, contas diferentes ainda podem apresentar
conjuntos de atributos numéricos idênticos.

Será investigada a presença de perfis transacionais
repetidos entre os conjuntos de treinamento e validação.

A auditoria será realizada nos cenários A, C e D,
considerando seus respectivos conjuntos de features.

O objetivo é verificar uma possível fonte de avaliação
otimista, sem modificar os conjuntos de dados nesta etapa.

O conjunto de teste permanecerá reservado.


In [27]:

import pandas as pd

# ==========================================
# Auditoria de perfis idênticos entre
# treinamento e validação - Vagifa
# ==========================================

resultados_perfis = []

for nome in [
    "A_Completo",
    "C_Sem_Contratos",
    "D_Atividade_Relacionamentos"
]:

    features = cenarios_vagifa[nome]

    treino = X_train_vagifa[features]
    validacao = X_val_vagifa[features]

    # Identificar perfis pelos valores das features
    hash_treino = pd.util.hash_pandas_object(
        treino, index=False
    )

    hash_validacao = pd.util.hash_pandas_object(
        validacao, index=False
    )

    # Verificar se perfis da validação aparecem no treino
    perfis_treino = set(hash_treino.tolist())

    repetidos_validacao = hash_validacao.isin(
        perfis_treino
    )

    resultados_perfis.append({
        "Cenario": nome,
        "Features": len(features),
        "Duplicatas_internas_treino": int(
            treino.duplicated().sum()
        ),
        "Perfis_validacao_ja_presentes_treino": int(
            repetidos_validacao.sum()
        ),
        "Percentual_validacao_repetida": round(
            repetidos_validacao.mean() * 100, 2
        )
    })

df_perfis = pd.DataFrame(resultados_perfis)

print("=== AUDITORIA DE PERFIS REPETIDOS ===")
display(df_perfis)


=== AUDITORIA DE PERFIS REPETIDOS ===


,Cenario,Features,Duplicatas_internas_treino,Perfis_validacao_ja_presentes_treino,Percentual_validacao_repetida
0,A_Completo,22,330,115,5.86
1,C_Sem_Contratos,18,330,115,5.86
2,D_Atividade_Relacionamentos,6,3090,1141,58.13



## 22 - Auditoria de conflitos entre perfis transacionais

A comparação entre treinamento e validação identificou
perfis numéricos idênticos entre contas Ethereum distintas.

No Cenário A, 5,86% dos registros de validação possuem
perfis já presentes no treinamento.

No Cenário D, esse percentual aumenta para 58,13%.

Será investigado se perfis idênticos apresentam
rótulos de fraude diferentes, tanto internamente
no treinamento quanto entre treinamento e validação.

O objetivo é avaliar possíveis ambiguidades nos dados
e limitações do particionamento aleatório.

O conjunto de teste permanecerá reservado.


In [28]:

import pandas as pd
import numpy as np

# ==========================================
# Conflitos de rótulos em perfis idênticos
# ==========================================

resultados_conflitos = []

for nome in [
    "A_Completo",
    "C_Sem_Contratos",
    "D_Atividade_Relacionamentos"
]:

    features = cenarios_vagifa[nome]

    # Assinaturas dos perfis numéricos
    hash_treino = pd.util.hash_pandas_object(
        X_train_vagifa[features],
        index=False
    ).to_numpy()

    hash_val = pd.util.hash_pandas_object(
        X_val_vagifa[features],
        index=False
    ).to_numpy()

    # Associar cada perfil ao rótulo no treinamento
    perfis_treino = pd.DataFrame({
        "perfil": hash_treino,
        "FLAG": y_train_vagifa.to_numpy()
    })

    resumo_treino = (
        perfis_treino.groupby("perfil")["FLAG"]
        .agg(["min", "max"])
    )

    # Um perfil com min diferente de max
    # aparece nas duas classes
    conflitos_treino = (
        resumo_treino["min"] != resumo_treino["max"]
    ).sum()

    # Verificar os perfis vistos na validação
    vistos = pd.Series(hash_val).isin(
        resumo_treino.index
    ).to_numpy()

    rotulos_val = y_val_vagifa.to_numpy()

    # Identificar perfis cujo rótulo da validação
    # não apareceu no treinamento para aquele perfil
    perfis_vistos = hash_val[vistos]

    minimo = resumo_treino["min"].reindex(
        perfis_vistos
    ).to_numpy()

    maximo = resumo_treino["max"].reindex(
        perfis_vistos
    ).to_numpy()

    rotulos_vistos = rotulos_val[vistos]

    conflitos_validacao = (
        (rotulos_vistos < minimo) |
        (rotulos_vistos > maximo)
    ).sum()

    resultados_conflitos.append({
        "Cenario": nome,
        "Perfis_unicos_treino": len(resumo_treino),
        "Perfis_treino_ambas_classes": int(
            conflitos_treino
        ),
        "Registros_validacao_vistos": int(
            vistos.sum()
        ),
        "Rotulos_incompativeis_validacao": int(
            conflitos_validacao
        ),
        "Registros_validacao_novos": int(
            (~vistos).sum()
        )
    })

df_conflitos_perfis = pd.DataFrame(
    resultados_conflitos
)

display(df_conflitos_perfis)


,Cenario,Perfis_unicos_treino,Perfis_treino_ambas_classes,Registros_validacao_vistos,Rotulos_incompativeis_validacao,Registros_validacao_novos
0,A_Completo,5559,3,115,1,1848
1,C_Sem_Contratos,5559,3,115,1,1848
2,D_Atividade_Relacionamentos,2799,84,1141,26,822



## 23 - Avaliação do XGBoost em perfis novos e já vistos

A auditoria identificou perfis numéricos idênticos entre
treinamento e validação, além de perfis associados às
duas classes de fraude.

Nesta etapa, será avaliado separadamente o desempenho
do XGBoost em:

- Registros cujos perfis já apareceram no treinamento;
- Registros com perfis não observados no treinamento.

Serão comparadas a quantidade de registros, a prevalência
de fraude, Average Precision, Precision, Recall e F1.

Essa avaliação permitirá investigar se o desempenho
dos modelos depende da repetição de perfis transacionais.

O conjunto de teste permanecerá reservado.


In [29]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# XGBoost - Perfis vistos versus novos
# ==========================================

resultados_perfis_xgb = []

for nome in [
    "A_Completo",
    "C_Sem_Contratos",
    "D_Atividade_Relacionamentos"
]:

    features = cenarios_vagifa[nome]

    # Identificar perfis presentes no treinamento
    hash_treino = set(
        pd.util.hash_pandas_object(
            X_train_vagifa[features],
            index=False
        ).to_numpy()
    )

    hash_validacao = pd.util.hash_pandas_object(
        X_val_vagifa[features],
        index=False
    ).to_numpy()

    perfil_visto = pd.Series(
        hash_validacao
    ).isin(hash_treino).to_numpy()

    # Modelo XGBoost previamente treinado
    modelo = modelos_xgb_vagifa[nome]

    probabilidades = modelo.predict_proba(
        X_val_vagifa[features]
    )[:, 1]

    rotulos = y_val_vagifa.to_numpy()

    # Avaliar separadamente os dois grupos
    for grupo, mascara in [
        ("Perfil já visto", perfil_visto),
        ("Perfil novo", ~perfil_visto)
    ]:

        y_grupo = rotulos[mascara]
        prob_grupo = probabilidades[mascara]

        pred_grupo = (prob_grupo >= 0.5).astype(int)

        # AP exige cautela quando há somente uma classe
        ap = (
            average_precision_score(
                y_grupo, prob_grupo
            )
            if len(np.unique(y_grupo)) == 2
            else np.nan
        )

        resultados_perfis_xgb.append({
            "Cenario": nome,
            "Grupo": grupo,
            "Registros": len(y_grupo),
            "Fraudes": int(y_grupo.sum()),
            "Prevalencia_fraude": y_grupo.mean(),
            "AP": ap,
            "Precision": precision_score(
                y_grupo, pred_grupo, zero_division=0
            ),
            "Recall": recall_score(
                y_grupo, pred_grupo, zero_division=0
            ),
            "F1": f1_score(
                y_grupo, pred_grupo, zero_division=0
            )
        })

df_resultados_perfis_xgb = pd.DataFrame(
    resultados_perfis_xgb
)

display(df_resultados_perfis_xgb.round(4))


,Cenario,Grupo,Registros,Fraudes,Prevalencia_fraude,AP,Precision,Recall,F1
0,A_Completo,Perfil já visto,115,109,0.9478,0.9986,0.9550,0.9725,0.9636
1,A_Completo,Perfil novo,1848,327,0.1769,0.9141,0.8951,0.7309,0.8047
2,C_Sem_Contratos,Perfil já visto,115,109,0.9478,0.9986,0.9550,0.9725,0.9636
3,C_Sem_Contratos,Perfil novo,1848,327,0.1769,0.9140,0.8930,0.7401,0.8094
4,D_Atividade_Relacionamentos,Perfil já visto,1141,308,0.2699,0.8502,0.9482,0.5942,0.7305
5,D_Atividade_Relacionamentos,Perfil novo,822,128,0.1557,0.8063,0.8660,0.6562,0.7467


In [30]:

# ==========================================
# Perfis repetidos - Cenário C
# ==========================================

features = cenarios_vagifa["C_Sem_Contratos"]

treino = X_train_vagifa[features].copy()
treino["FLAG"] = y_train_vagifa.to_numpy()
treino["Conjunto"] = "Treino"

validacao = X_val_vagifa[features].copy()
validacao["FLAG"] = y_val_vagifa.to_numpy()
validacao["Conjunto"] = "Validacao"

df_perfis_auditoria = pd.concat(
    [treino, validacao],
    ignore_index=True
)

# Identificador de perfil pelas features
df_perfis_auditoria["perfil"] = (
    pd.util.hash_pandas_object(
        df_perfis_auditoria[features],
        index=False
    )
)

# Resumo por perfil
resumo_perfis = (
    df_perfis_auditoria
    .groupby("perfil")
    .agg(
        registros=("FLAG", "size"),
        fraudes=("FLAG", "sum"),
        treino=("Conjunto", lambda x: (x == "Treino").sum()),
        validacao=("Conjunto", lambda x: (x == "Validacao").sum())
    )
)

resumo_perfis["percentual_fraude"] = (
    resumo_perfis["fraudes"]
    / resumo_perfis["registros"] * 100
)

# Perfis presentes em ambos os conjuntos
perfis_compartilhados = resumo_perfis[
    (resumo_perfis["treino"] > 0) &
    (resumo_perfis["validacao"] > 0)
].sort_values(
    ["validacao", "registros"],
    ascending=False
)

print("=== PERFIS COMPARTILHADOS ===")
print("Quantidade de perfis:", len(perfis_compartilhados))

display(perfis_compartilhados.head(15))

print("\n=== PERFIL MAIS FREQUENTE ===")

if len(perfis_compartilhados) > 0:
    perfil_principal = perfis_compartilhados.index[0]

    display(
        df_perfis_auditoria.loc[
            df_perfis_auditoria["perfil"] == perfil_principal,
            features + ["FLAG", "Conjunto"]
        ].head(10)
    )


=== PERFIS COMPARTILHADOS ===
Quantidade de perfis: 7


,registros,fraudes,treino,validacao,percentual_fraude
perfil,,,,,
3057750390331765523,427,427,322,105,100.000000
16718483348058156896,4,4,1,3,100.000000
3505593638406641304,4,1,2,2,25.000000
3791756682666024423,3,0,1,2,0.000000
626057525915053301,6,6,5,1,100.000000
725907192660956649,3,1,2,1,33.333333
5735138072584142455,2,1,1,1,50.000000



=== PERFIL MAIS FREQUENTE ===


,Avg min between sent tnx,Avg min between received tnx,Time Diff between first and last (Mins),Sent tnx,Received Tnx,Number of Created Contracts,Unique Received From Addresses,Unique Sent To Addresses,min value received,max value received,avg val received,min val sent,max val sent,avg val sent,total transactions (including tnx to create contract,total Ether sent,total ether received,total ether balance,FLAG,Conjunto
36,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
72,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
78,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
100,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
119,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
138,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
153,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
163,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
169,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino
195,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,1,Treino



## 24 - Investigação dos perfis totalmente zerados

A análise identificou um perfil com todas as 18 features
do Cenário C iguais a zero.

Esse perfil apareceu em 427 registros compartilhados entre
treinamento e validação, todos classificados como fraude.

Serão investigadas as possíveis relações entre esse padrão,
a ausência dos atributos ERC-20 e a variável-alvo FLAG.

O objetivo é verificar se o modelo pode estar utilizando
a ausência de atividade registrada como um atalho de
classificação.

A análise utilizará apenas treinamento e validação.
O conjunto de teste permanecerá reservado.


In [31]:

import pandas as pd

# ==========================================
# Perfis zerados e ausência de atributos ERC-20
# ==========================================

features = cenarios_vagifa["C_Sem_Contratos"]

# Colunas ERC-20 preservadas na base preparada
colunas_erc20 = [
    c for c in df_vagifa_preparado.columns
    if "erc20" in c.lower()
]

resultados_zero = []

for nome, X_conjunto, y_conjunto in [
    ("Treinamento", X_train_vagifa, y_train_vagifa),
    ("Validação", X_val_vagifa, y_val_vagifa)
]:

    # Todos os atributos do cenário C zerados
    perfil_zero = (
        X_conjunto[features] == 0
    ).all(axis=1)

    # Todos os atributos ERC-20 numéricos ausentes
    erc20_ausente = (
        df_vagifa_preparado
        .loc[X_conjunto.index, colunas_erc20]
        .isna()
        .all(axis=1)
    )

    resultados_zero.append({
        "Conjunto": nome,
        "Perfis_totalmente_zero": int(perfil_zero.sum()),
        "Fraudes_entre_zeros": int(
            y_conjunto.loc[perfil_zero].sum()
        ),
        "Zero_e_ERC20_ausente": int(
            (perfil_zero & erc20_ausente).sum()
        ),
        "Zero_com_ERC20_presente": int(
            (perfil_zero & ~erc20_ausente).sum()
        ),
        "ERC20_ausente_total": int(erc20_ausente.sum())
    })

    print(f"\n=== {nome} ===")

    display(
        pd.crosstab(
            perfil_zero.rename("Perfil_totalmente_zero"),
            y_conjunto.rename("FLAG")
        )
    )

df_analise_zero = pd.DataFrame(resultados_zero)

print("\n=== RELAÇÃO ENTRE PERFIL ZERO E ERC-20 ===")
display(df_analise_zero)



=== Treinamento ===


FLAG,0,1
Perfil_totalmente_zero,,
False,4582,985
True,0,322



=== Validação ===


FLAG,0,1
Perfil_totalmente_zero,,
False,1527,331
True,0,105



=== RELAÇÃO ENTRE PERFIL ZERO E ERC-20 ===


,Conjunto,Perfis_totalmente_zero,Fraudes_entre_zeros,Zero_e_ERC20_ausente,Zero_com_ERC20_presente,ERC20_ausente_total
0,Treinamento,322,322,157,165,500
1,Validação,105,105,59,46,173



## 25 - Sensibilidade do XGBoost aos perfis zerados

A auditoria identificou 427 registros com todas as
18 features do Cenário C iguais a zero, distribuídos
entre treinamento e validação.

Todos esses registros pertencem à classe FLAG = 1.

Foi observado que aproximadamente metade deles
também apresenta ausência dos atributos ERC-20,
enquanto a outra metade possui informações ERC-20.

Será realizada uma análise de sensibilidade do
XGBoost previamente treinado, comparando seu
desempenho na validação completa com o desempenho
nos registros sem perfil totalmente zerado.

Serão utilizadas Average Precision, Precision,
Recall, F1 e matriz de confusão.

As prevalências de fraude serão apresentadas para
permitir uma comparação adequada.

O conjunto de teste permanecerá reservado.


In [32]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Cenário C - 18 features
features = cenarios_vagifa["C_Sem_Contratos"]

# Modelo XGBoost já treinado
modelo = modelos_xgb_vagifa["C_Sem_Contratos"]

# Dados de validação
X_val = X_val_vagifa[features]
y_val = y_val_vagifa

# Identificar perfis totalmente zerados
perfil_zero = (X_val == 0).all(axis=1)

# Probabilidades geradas pelo modelo
probabilidades = modelo.predict_proba(X_val)[:, 1]

resultados_sensibilidade = []

for nome, mascara in [
    ("Validação completa", np.ones(len(X_val), dtype=bool)),
    ("Sem perfis zerados", (~perfil_zero).to_numpy())
]:

    y_grupo = y_val.to_numpy()[mascara]
    prob_grupo = probabilidades[mascara]

    pred_grupo = (prob_grupo >= 0.5).astype(int)

    matriz = confusion_matrix(
        y_grupo,
        pred_grupo,
        labels=[0, 1]
    )

    resultados_sensibilidade.append({
        "Grupo": nome,
        "Registros": len(y_grupo),
        "Fraudes": int(y_grupo.sum()),
        "Prevalencia": y_grupo.mean(),
        "AP": average_precision_score(
            y_grupo, prob_grupo
        ),
        "Precision": precision_score(
            y_grupo, pred_grupo, zero_division=0
        ),
        "Recall": recall_score(
            y_grupo, pred_grupo, zero_division=0
        ),
        "F1": f1_score(
            y_grupo, pred_grupo, zero_division=0
        )
    })

    print(f"\n=== {nome} ===")
    print("Matriz de confusão:")
    print(matriz)

df_sensibilidade_zero = pd.DataFrame(
    resultados_sensibilidade
)

print("\n=== COMPARAÇÃO ===")
display(df_sensibilidade_zero.round(4))



=== Validação completa ===
Matriz de confusão:
[[1493   34]
 [  88  348]]

=== Sem perfis zerados ===
Matriz de confusão:
[[1493   34]
 [  88  243]]

=== COMPARAÇÃO ===


,Grupo,Registros,Fraudes,Prevalencia,AP,Precision,Recall,F1
0,Validação completa,1963,436,0.2221,0.9430,0.9110,0.7982,0.8509
1,Sem perfis zerados,1858,331,0.1781,0.9038,0.8773,0.7341,0.7993



## 26 - Retreinamento do XGBoost sem perfis zerados

Após identificar a influência dos perfis totalmente
zerados, será treinado um segundo XGBoost excluindo
esses registros do conjunto de treinamento.

O modelo original e o novo modelo serão avaliados
sobre o mesmo subconjunto de validação, composto
exclusivamente por registros com pelo menos uma
feature diferente de zero no Cenário C.

Serão mantidos os mesmos hiperparâmetros e o
threshold de classificação de 0,5.

Essa análise de sensibilidade busca avaliar o impacto
da presença dos perfis zerados no treinamento.

O conjunto de teste permanece reservado.


In [33]:

from xgboost import XGBClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

# ==========================================
# XGBoost: com e sem perfis zerados no treino
# ==========================================

features = cenarios_vagifa["C_Sem_Contratos"]

# Identificar perfis totalmente zerados
zero_treino = X_train_vagifa[features].eq(0).all(axis=1)
zero_val = X_val_vagifa[features].eq(0).all(axis=1)

# Dados de treino sem perfis zerados
X_treino_filtrado = X_train_vagifa.loc[
    ~zero_treino, features
]

y_treino_filtrado = y_train_vagifa.loc[
    ~zero_treino
]

# Mesma validação para os dois modelos
X_val_filtrado = X_val_vagifa.loc[
    ~zero_val, features
]

y_val_filtrado = y_val_vagifa.loc[
    ~zero_val
]

print("=== TAMANHO DOS CONJUNTOS ===")
print("Treino original:", len(X_train_vagifa))
print("Perfis zerados removidos:", zero_treino.sum())
print("Treino filtrado:", len(X_treino_filtrado))
print("Validação filtrada:", len(X_val_filtrado))

# Modelo original previamente treinado
modelo_original = modelos_xgb_vagifa["C_Sem_Contratos"]

# Novo modelo com os mesmos hiperparâmetros
modelo_sem_zeros = XGBClassifier(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    tree_method="hist"
)

modelo_sem_zeros.fit(
    X_treino_filtrado,
    y_treino_filtrado
)

resultados_comparacao_zero = []

for nome, modelo in [
    ("Treinado com perfis zerados", modelo_original),
    ("Treinado sem perfis zerados", modelo_sem_zeros)
]:

    probabilidades = modelo.predict_proba(
        X_val_filtrado
    )[:, 1]

    predicoes = (probabilidades >= 0.5).astype(int)

    resultados_comparacao_zero.append({
        "Modelo": nome,
        "Registros_validacao": len(y_val_filtrado),
        "Fraudes_validacao": int(y_val_filtrado.sum()),
        "AP": average_precision_score(
            y_val_filtrado, probabilidades
        ),
        "Precision": precision_score(
            y_val_filtrado, predicoes, zero_division=0
        ),
        "Recall": recall_score(
            y_val_filtrado, predicoes, zero_division=0
        ),
        "F1": f1_score(
            y_val_filtrado, predicoes, zero_division=0
        )
    })

    print(f"\n=== {nome} ===")
    print(
        confusion_matrix(
            y_val_filtrado, predicoes, labels=[0, 1]
        )
    )

df_comparacao_zero = pd.DataFrame(
    resultados_comparacao_zero
)

print("\n=== COMPARAÇÃO FINAL ===")
display(df_comparacao_zero.round(4))


=== TAMANHO DOS CONJUNTOS ===
Treino original: 5889
Perfis zerados removidos: 322
Treino filtrado: 5567
Validação filtrada: 1858

=== Treinado com perfis zerados ===
[[1493   34]
 [  88  243]]

=== Treinado sem perfis zerados ===
[[1496   31]
 [  84  247]]

=== COMPARAÇÃO FINAL ===


,Modelo,Registros_validacao,Fraudes_validacao,AP,Precision,Recall,F1
0,Treinado com perfis zerados,1858,331,0.9038,0.8773,0.7341,0.7993
1,Treinado sem perfis zerados,1858,331,0.9117,0.8885,0.7462,0.8112



## 27 - Validação cruzada agrupada por perfil

Será realizada uma comparação entre validação cruzada
estratificada convencional e validação cruzada
estratificada agrupada por perfil numérico.

Serão utilizados exclusivamente os registros não zerados
do conjunto de treinamento do Cenário C.

Na validação agrupada, registros com as mesmas 18 features
serão mantidos no mesmo fold, evitando compartilhamento
de perfis idênticos entre treino e validação.

O objetivo é avaliar o impacto da repetição dos perfis
sobre o desempenho do XGBoost.

Serão calculadas Average Precision, F1, Recall, Precision
e prevalência de fraude em cada fold.

Os conjuntos de validação original e teste final
permanecerão fora deste experimento.


In [34]:

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold
)
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# Apenas o treinamento sem perfis zerados
X_cv = X_treino_filtrado.reset_index(drop=True)
y_cv = y_treino_filtrado.reset_index(drop=True)

# Identificar perfis idênticos pelas 18 features
grupos = pd.util.hash_pandas_object(
    X_cv, index=False
).to_numpy()

divisoes = {
    "Estratificada": StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42
    ),
    "Agrupada_por_perfil": StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=42
    )
}

resultados_cv = []

for estrategia, divisor in divisoes.items():

    if estrategia == "Agrupada_por_perfil":
        folds = divisor.split(X_cv, y_cv, groups=grupos)
    else:
        folds = divisor.split(X_cv, y_cv)

    for numero_fold, (idx_treino, idx_val) in enumerate(
        folds, start=1
    ):

        X_fold_treino = X_cv.iloc[idx_treino]
        X_fold_val = X_cv.iloc[idx_val]
        y_fold_treino = y_cv.iloc[idx_treino]
        y_fold_val = y_cv.iloc[idx_val]

        modelo_cv = XGBClassifier(
            n_estimators=150,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            eval_metric="logloss",
            tree_method="hist"
        )

        modelo_cv.fit(X_fold_treino, y_fold_treino)

        probabilidades = modelo_cv.predict_proba(
            X_fold_val
        )[:, 1]

        predicoes = (probabilidades >= 0.5).astype(int)

        grupos_treino = set(grupos[idx_treino])
        grupos_val = set(grupos[idx_val])

        grupos_compartilhados = len(
            grupos_treino.intersection(grupos_val)
        )

        resultados_cv.append({
            "Estrategia": estrategia,
            "Fold": numero_fold,
            "Registros_validacao": len(idx_val),
            "Prevalencia": y_fold_val.mean(),
            "Grupos_compartilhados": grupos_compartilhados,
            "AP": average_precision_score(
                y_fold_val, probabilidades
            ),
            "Precision": precision_score(
                y_fold_val, predicoes, zero_division=0
            ),
            "Recall": recall_score(
                y_fold_val, predicoes, zero_division=0
            ),
            "F1": f1_score(
                y_fold_val, predicoes, zero_division=0
            )
        })

df_cv_perfis = pd.DataFrame(resultados_cv)

print("=== RESULTADOS POR FOLD ===")
display(df_cv_perfis.round(4))

print("\n=== MÉDIAS POR ESTRATÉGIA ===")

resumo_cv = df_cv_perfis.groupby("Estrategia").agg(
    AP_media=("AP", "mean"),
    AP_desvio=("AP", "std"),
    F1_medio=("F1", "mean"),
    Recall_medio=("Recall", "mean"),
    Precision_media=("Precision", "mean"),
    Prevalencia_media=("Prevalencia", "mean"),
    Grupos_compartilhados_total=("Grupos_compartilhados", "sum")
)

display(resumo_cv.round(4))


=== RESULTADOS POR FOLD ===


,Estrategia,Fold,Registros_validacao,Prevalencia,Grupos_compartilhados,AP,Precision,Recall,F1
0,Estratificada,1,1114,0.1768,2,0.9211,0.8671,0.6954,0.7718
1,Estratificada,2,1114,0.1768,3,0.8935,0.8608,0.6904,0.7662
2,Estratificada,3,1113,0.1770,3,0.9405,0.8703,0.8173,0.8429
3,Estratificada,4,1113,0.1770,1,0.9242,0.8953,0.7817,0.8347
4,Estratificada,5,1113,0.1770,1,0.9089,0.8848,0.7411,0.8066
5,Agrupada_por_perfil,1,1113,0.2013,0,0.9298,0.8973,0.7411,0.8117
6,Agrupada_por_perfil,2,1112,0.1835,0,0.9059,0.8837,0.7451,0.8085
7,Agrupada_por_perfil,3,1114,0.1553,0,0.9125,0.8844,0.7514,0.8125
8,Agrupada_por_perfil,4,1114,0.1607,0,0.8911,0.8387,0.7263,0.7784
9,Agrupada_por_perfil,5,1114,0.1840,0,0.9436,0.9186,0.7707,0.8382



=== MÉDIAS POR ESTRATÉGIA ===


,AP_media,AP_desvio,F1_medio,Recall_medio,Precision_media,Prevalencia_media,Grupos_compartilhados_total
Estrategia,,,,,,,
Agrupada_por_perfil,0.9166,0.0205,0.8099,0.7469,0.8845,0.1769,0
Estratificada,0.9176,0.0176,0.8045,0.7452,0.8757,0.1769,10



## 28 - Investigação da origem dos rótulos do Vagifa

Será investigada a relação entre o dataset Vagifa
disponibilizado no Kaggle e a base publicada por
Farrugia, Ellul e Azzopardi (2020).

A pesquisa original utiliza endereços identificados
como ilícitos pela comunidade Ethereum e estatísticas
extraídas do histórico de transações.

Serão comparados os endereços e rótulos dos conjuntos
de treinamento e validação com a base original.

O objetivo é verificar a correspondência entre as
fontes e identificar possíveis diferenças na
composição das classes.

O conjunto de teste final permanecerá reservado.


In [35]:

import pandas as pd

# Dataset publicado no projeto original
url_origem = (
    "https://raw.githubusercontent.com/"
    "sfarrugia15/Ethereum_Fraud_Detection/"
    "master/Account_Stats/Complete.csv"
)

df_farrugia = pd.read_csv(url_origem)

df_farrugia.columns = df_farrugia.columns.str.strip()

df_farrugia["Address_norm"] = (
    df_farrugia["Address"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("=== BASE ORIGINAL DE FARRUGIA ===")
print("Dimensões:", df_farrugia.shape)
print(df_farrugia["FLAG"].value_counts())

# Reunir somente treinamento e validação do Vagifa
df_auditoria_origem = pd.concat([
    df_vagifa_preparado.loc[
        X_train_vagifa.index, ["Address_norm", "FLAG"]
    ].assign(Conjunto="Treinamento"),

    df_vagifa_preparado.loc[
        X_val_vagifa.index, ["Address_norm", "FLAG"]
    ].assign(Conjunto="Validação")
])

# Verificar conflitos internos na base publicada
conflitos_origem = (
    df_farrugia.groupby("Address_norm")["FLAG"]
    .nunique()
    .gt(1)
    .sum()
)

print("\nConflitos de rótulo na base original:",
      conflitos_origem)

# Mapear endereço para classificação original
mapa_origem = (
    df_farrugia
    .drop_duplicates(subset=["Address_norm"])
    .set_index("Address_norm")["FLAG"]
)

df_auditoria_origem["FLAG_origem"] = (
    df_auditoria_origem["Address_norm"].map(mapa_origem)
)

df_auditoria_origem["Encontrado_na_origem"] = (
    df_auditoria_origem["FLAG_origem"].notna()
)

df_auditoria_origem["Rotulo_divergente"] = (
    df_auditoria_origem["Encontrado_na_origem"]
    & (
        df_auditoria_origem["FLAG"]
        != df_auditoria_origem["FLAG_origem"]
    )
)

print("\n=== CORRESPONDÊNCIA COM A BASE ORIGINAL ===")

display(pd.crosstab(
    df_auditoria_origem["FLAG"],
    df_auditoria_origem["Encontrado_na_origem"],
    margins=True
))

print("\n=== RESUMO ===")
print(
    "Endereços encontrados:",
    df_auditoria_origem["Encontrado_na_origem"].sum()
)
print(
    "Rótulos divergentes:",
    df_auditoria_origem["Rotulo_divergente"].sum()
)


=== BASE ORIGINAL DE FARRUGIA ===
Dimensões: (4681, 51)
FLAG
0    2502
1    2179
Name: count, dtype: int64

Conflitos de rótulo na base original: 1

=== CORRESPONDÊNCIA COM A BASE ORIGINAL ===


Encontrado_na_origem,False,True,All
FLAG,,,
0,4481,1628,6109
1,0,1743,1743
All,4481,3371,7852



=== RESUMO ===
Endereços encontrados: 3371
Rótulos divergentes: 0



## 29 - Investigação de conflitos de rotulação na fonte original

A comparação do Vagifa com a base publicada por Farrugia
identificou correspondência de todos os endereços
fraudulentos presentes no treinamento e validação.

Entretanto, a base original apresentou um endereço
associado a mais de um rótulo.

Será investigado esse conflito antes de concluir a
auditoria de proveniência.

O conjunto de teste permanecerá reservado.


In [39]:

# ==========================================
# Conflitos de rótulo na base original
# ==========================================

contagem_rotulos = (
    df_farrugia.groupby("Address_norm")["FLAG"]
    .nunique()
)

enderecos_conflitantes = contagem_rotulos[
    contagem_rotulos > 1
].index

print("=== ENDEREÇOS CONFLITANTES ===")
print("Quantidade:", len(enderecos_conflitantes))

display(
    df_farrugia.loc[
        df_farrugia["Address_norm"].isin(
            enderecos_conflitantes
        ),
        ["Address", "FLAG"]
    ]
)

print("\n=== PRESENÇA NO VAGIFA (TREINO + VALIDAÇÃO) ===")

display(
    df_auditoria_origem.loc[
        df_auditoria_origem["Address_norm"].isin(
            enderecos_conflitantes
        ),
        ["Address_norm", "FLAG", "Conjunto", "FLAG_origem"]
    ]
)


=== ENDEREÇOS CONFLITANTES ===
Quantidade: 1


,Address,FLAG
2178,0xd624d046edbdef805c5e4140dce5fb5ec1b39a3c,1
3220,0xd624d046edbdef805c5e4140dce5fb5ec1b39a3c,0



=== PRESENÇA NO VAGIFA (TREINO + VALIDAÇÃO) ===


,Address_norm,FLAG,Conjunto,FLAG_origem



## 30 - Investigação de possível viés de seleção

A análise de proveniência identificou que todas as contas
fraudulentas analisadas possuem correspondência com a
base original de Farrugia.

Entretanto, uma parcela significativa das contas não
fraudulentas não foi encontrada nessa fonte.

Será avaliado o comportamento do XGBoost separadamente
nos dois grupos de contas não fraudulentas.

Serão comparadas as probabilidades atribuídas pelo modelo,
a quantidade de falsos positivos e a taxa de falsos positivos.

O objetivo é investigar possíveis diferenças relacionadas
aos procedimentos de seleção e composição do dataset.

O conjunto de teste permanecerá reservado.


In [40]:

import pandas as pd
import numpy as np

# ==========================================
# Auditoria de viés de seleção - Vagifa
# ==========================================

features = cenarios_vagifa["C_Sem_Contratos"]

# Modelo original do cenário C
modelo = modelos_xgb_vagifa["C_Sem_Contratos"]

# Informações de proveniência da validação
df_vies = df_auditoria_origem.loc[
    X_val_vagifa.index
].copy()

# Garantir o alinhamento dos registros
assert df_vies.index.equals(X_val_vagifa.index)

# Probabilidades e predições
df_vies["Probabilidade_fraude"] = (
    modelo.predict_proba(
        X_val_vagifa[features]
    )[:, 1]
)

df_vies["Predicao"] = (
    df_vies["Probabilidade_fraude"] >= 0.5
).astype(int)

# Analisar somente contas rotuladas como não fraudulentas
df_normais = df_vies[
    df_vies["FLAG"] == 0
].copy()

df_normais["Grupo_origem"] = np.where(
    df_normais["Encontrado_na_origem"],
    "Encontrado em Farrugia",
    "Não encontrado em Farrugia"
)

# Comparar o comportamento do modelo
resumo_vies = df_normais.groupby("Grupo_origem").agg(
    Registros=("FLAG", "size"),
    Probabilidade_media=("Probabilidade_fraude", "mean"),
    Probabilidade_mediana=("Probabilidade_fraude", "median"),
    Falsos_positivos=("Predicao", "sum")
)

resumo_vies["Taxa_falsos_positivos_%"] = (
    resumo_vies["Falsos_positivos"]
    / resumo_vies["Registros"] * 100
)

print("=== COMPARAÇÃO POR GRUPO DE ORIGEM ===")

display(resumo_vies.round(4))

print("\n=== DISTRIBUIÇÃO DA PROVENIÊNCIA NA VALIDAÇÃO ===")

display(pd.crosstab(
    df_vies["FLAG"],
    df_vies["Encontrado_na_origem"],
    margins=True
))


=== COMPARAÇÃO POR GRUPO DE ORIGEM ===


,Registros,Probabilidade_media,Probabilidade_mediana,Falsos_positivos,Taxa_falsos_positivos_%
Grupo_origem,,,,,
Encontrado em Farrugia,368,0.0484,0.0123,3,0.8152
Não encontrado em Farrugia,1159,0.0697,0.0129,31,2.6747



=== DISTRIBUIÇÃO DA PROVENIÊNCIA NA VALIDAÇÃO ===


Encontrado_na_origem,False,True,All
FLAG,,,
0,1159,368,1527
1,0,436,436
All,1159,804,1963



## 31 - Definição dos thresholds finais

Após a conclusão dos experimentos exploratórios,
foram selecionadas configurações candidatas da
Regressão Logística e do XGBoost.

Os thresholds de classificação serão definidos
exclusivamente utilizando o conjunto de validação,
adotando o maior F1-Score como critério.

A Average Precision continuará sendo utilizada
como principal métrica de capacidade discriminativa.

O conjunto de teste não será utilizado para
selecionar thresholds, features ou hiperparâmetros.


In [41]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# ==========================================
# Configurações finais candidatas - Vagifa
# ==========================================

modelos_finais_vagifa = {
    "Regressao_Logistica": {
        "modelo": modelos_logistica_vagifa[
            "D_Atividade_Relacionamentos"
        ],
        "cenario": "D_Atividade_Relacionamentos"
    },

    "Regressao_Logistica_Balanceada": {
        "modelo": modelos_logistica_balanceada[
            "D_Atividade_Relacionamentos"
        ],
        "cenario": "D_Atividade_Relacionamentos"
    },

    "XGBoost": {
        "modelo": modelos_xgb_vagifa[
            "C_Sem_Contratos"
        ],
        "cenario": "C_Sem_Contratos"
    }
}

resultados_threshold_final = []

for nome, configuracao in modelos_finais_vagifa.items():

    modelo = configuracao["modelo"]
    cenario = configuracao["cenario"]
    features = cenarios_vagifa[cenario]

    # Probabilidades exclusivamente na validação
    probabilidades = modelo.predict_proba(
        X_val_vagifa[features]
    )[:, 1]

    precision, recall, thresholds = precision_recall_curve(
        y_val_vagifa,
        probabilidades
    )

    # Calcular F1 para cada threshold disponível
    f1_thresholds = np.divide(
        2 * precision[:-1] * recall[:-1],
        precision[:-1] + recall[:-1],
        out=np.zeros_like(precision[:-1]),
        where=(precision[:-1] + recall[:-1]) != 0
    )

    melhor_indice = np.argmax(f1_thresholds)

    melhor_threshold = float(
        thresholds[melhor_indice]
    )

    predicoes = (
        probabilidades >= melhor_threshold
    ).astype(int)

    # Armazenar configuração para o teste final
    configuracao["threshold"] = melhor_threshold
    configuracao["features"] = features

    resultados_threshold_final.append({
        "Modelo": nome,
        "Cenario": cenario,
        "Features": len(features),
        "Threshold": melhor_threshold,
        "Precision": precision_score(
            y_val_vagifa, predicoes, zero_division=0
        ),
        "Recall": recall_score(
            y_val_vagifa, predicoes, zero_division=0
        ),
        "F1": f1_score(
            y_val_vagifa, predicoes, zero_division=0
        ),
        "AP": average_precision_score(
            y_val_vagifa, probabilidades
        )
    })

df_threshold_final = pd.DataFrame(
    resultados_threshold_final
)

print("=== CONFIGURAÇÕES SELECIONADAS NA VALIDAÇÃO ===")

display(df_threshold_final.round(4))


=== CONFIGURAÇÕES SELECIONADAS NA VALIDAÇÃO ===


,Modelo,Cenario,Features,Threshold,Precision,Recall,F1,AP
0,Regressao_Logistica,D_Atividade_Relacionamentos,6,0.2734,0.5556,0.5963,0.5752,0.6099
1,Regressao_Logistica_Balanceada,D_Atividade_Relacionamentos,6,0.5649,0.5597,0.5917,0.5753,0.6094
2,XGBoost,C_Sem_Contratos,18,0.3950,0.8806,0.8624,0.8714,0.9430



## 32 - Avaliação final dos modelos no dataset Vagifa

Após a seleção dos cenários de features e dos thresholds
utilizando exclusivamente a validação, será realizada
a avaliação final no conjunto de teste reservado.

Serão comparadas três configurações:

- Regressão Logística original;
- Regressão Logística com pesos de classe;
- XGBoost.

Os modelos previamente treinados serão preservados,
assim como os thresholds selecionados na validação.

Serão calculadas Precision, Recall, F1-Score,
Average Precision, PR-AUC trapezoidal e matrizes
de confusão.

Nenhuma decisão de ajuste será tomada com base
nos resultados do conjunto de teste.


In [42]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve,
    auc,
    confusion_matrix
)

# ==========================================
# Avaliação final - Dataset Vagifa
# ==========================================

resultados_teste_vagifa = []
matrizes_teste_vagifa = {}

for nome, configuracao in modelos_finais_vagifa.items():

    modelo = configuracao["modelo"]
    features = configuracao["features"]
    threshold = configuracao["threshold"]

    # Conjunto de teste reservado
    X_teste = X_test_vagifa[features]
    y_teste = y_test_vagifa

    # Probabilidades e threshold previamente definido
    probabilidades = modelo.predict_proba(
        X_teste
    )[:, 1]

    predicoes = (
        probabilidades >= threshold
    ).astype(int)

    # Curva Precision-Recall
    precision_curva, recall_curva, _ = (
        precision_recall_curve(
            y_teste, probabilidades
        )
    )

    # Matriz de confusão
    matriz = confusion_matrix(
        y_teste, predicoes, labels=[0, 1]
    )

    vn, fp, fn, vp = matriz.ravel()

    matrizes_teste_vagifa[nome] = matriz

    resultados_teste_vagifa.append({
        "Modelo": nome,
        "Cenario": configuracao["cenario"],
        "Features": len(features),
        "Threshold": threshold,
        "Precision": precision_score(
            y_teste, predicoes, zero_division=0
        ),
        "Recall": recall_score(
            y_teste, predicoes, zero_division=0
        ),
        "F1": f1_score(
            y_teste, predicoes, zero_division=0
        ),
        "AP": average_precision_score(
            y_teste, probabilidades
        ),
        "PR_AUC": auc(
            recall_curva, precision_curva
        ),
        "VN": vn,
        "FP": fp,
        "FN": fn,
        "VP": vp
    })

    print(f"\n=== {nome} ===")
    print("Matriz de confusão:")
    print(matriz)

df_teste_final_vagifa = pd.DataFrame(
    resultados_teste_vagifa
)

print("\n=== RESULTADOS FINAIS - VAGIFA ===")

display(df_teste_final_vagifa.round(4))

print("\n=== INFORMAÇÕES DO TESTE ===")
print("Registros:", len(y_test_vagifa))
print("Fraudes:", y_test_vagifa.sum())
print("Prevalência:", y_test_vagifa.mean())



=== Regressao_Logistica ===
Matriz de confusão:
[[1313  215]
 [ 162  274]]

=== Regressao_Logistica_Balanceada ===
Matriz de confusão:
[[1318  210]
 [ 166  270]]

=== XGBoost ===
Matriz de confusão:
[[1463   65]
 [  72  364]]

=== RESULTADOS FINAIS - VAGIFA ===


,Modelo,Cenario,Features,Threshold,Precision,Recall,F1,AP,PR_AUC,VN,FP,FN,VP
0,Regressao_Logistica,D_Atividade_Relacionamentos,6,0.2734,0.5603,0.6284,0.5924,0.6185,0.6040,1313,215,162,274
1,Regressao_Logistica_Balanceada,D_Atividade_Relacionamentos,6,0.5649,0.5625,0.6193,0.5895,0.6257,0.6120,1318,210,166,270
2,XGBoost,C_Sem_Contratos,18,0.3950,0.8485,0.8349,0.8416,0.9393,0.9393,1463,65,72,364



=== INFORMAÇÕES DO TESTE ===
Registros: 1964
Fraudes: 436
Prevalência: 0.2219959266802444


In [43]:

from pathlib import Path

pasta = Path("/content/resultados_vagifa")
pasta.mkdir(exist_ok=True)

tabelas = {
    "resultados_teste_final": df_teste_final_vagifa,
    "thresholds_finais": df_threshold_final,
    "comparacao_modelos": comparacao_vagifa,
    "validacao_agrupada": df_cv_perfis,
    "resumo_validacao_agrupada": resumo_cv.reset_index(),
    "sensibilidade_perfis_zerados": df_sensibilidade_zero,
    "retreinamento_sem_zeros": df_comparacao_zero,
    "dicionario_features": df_glossario_vagifa,
    "analise_vies_selecao": resumo_vies.reset_index()
}

for nome, tabela in tabelas.items():

    tabela.to_csv(
        pasta / f"{nome}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("=== ARQUIVOS EXPORTADOS ===")

for arquivo in pasta.iterdir():
    print(arquivo.name)


=== ARQUIVOS EXPORTADOS ===
sensibilidade_perfis_zerados.csv
dicionario_features.csv
resultados_teste_final.csv
thresholds_finais.csv
resumo_validacao_agrupada.csv
comparacao_modelos.csv
validacao_agrupada.csv
retreinamento_sem_zeros.csv
analise_vies_selecao.csv



## 33 - Avaliação do desempenho computacional dos modelos

Após a avaliação preditiva final, será realizada a medição
do custo computacional de inferência dos modelos selecionados.

Serão avaliadas as seguintes métricas:

- Latência média de inferência;
- Latência mediana (p50);
- Latência p95;
- Latência p99;
- Throughput de inferência.

Para a avaliação individual, serão utilizados 500 registros
selecionados com semente aleatória fixa.

Para a avaliação em lote, serão realizadas 30 repetições
sobre os 1.964 registros do conjunto de teste.

Serão executadas inferências de aquecimento antes das medições.

O tempo medido incluirá o pré-processamento interno do
pipeline, a geração das probabilidades e a aplicação do
threshold previamente definido.

Não serão considerados os tempos de treinamento, coleta
dos dados ou extração original dos atributos da blockchain.

As características do ambiente computacional serão registradas
para auxiliar na interpretação e reprodução dos resultados.


In [44]:

import time
import platform
import os
import sys
import numpy as np
import pandas as pd
import sklearn
import xgboost

# ==========================================
# Ambiente computacional
# ==========================================

ambiente = {
    "Sistema": platform.platform(),
    "Arquitetura": platform.machine(),
    "CPUs_logicas": os.cpu_count(),
    "Python": platform.python_version(),
    "Scikit_learn": sklearn.__version__,
    "XGBoost": xgboost.__version__,
    "NumPy": np.__version__,
    "Pandas": pd.__version__
}

if os.path.exists("/proc/cpuinfo"):
    with open("/proc/cpuinfo") as arquivo:
        for linha in arquivo:
            if linha.startswith("model name"):
                ambiente["CPU"] = linha.split(":", 1)[1].strip()
                break

print("=== AMBIENTE COMPUTACIONAL ===")

for chave, valor in ambiente.items():
    print(f"{chave}: {valor}")

# ==========================================
# Configuração das medições
# ==========================================

N_INDIVIDUAL = 500
N_REPETICOES_LOTE = 30
N_AQUECIMENTO = 10

rng = np.random.default_rng(42)

indices = rng.choice(
    len(X_test_vagifa),
    size=N_INDIVIDUAL,
    replace=False
)

resultados_desempenho_vagifa = []

def inferencia(modelo, dados, threshold):
    probabilidades = modelo.predict_proba(dados)[:, 1]
    return (probabilidades >= threshold).astype(int)

# ==========================================
# Benchmark dos modelos finais
# ==========================================

for nome, config in modelos_finais_vagifa.items():

    modelo = config["modelo"]
    features = config["features"]
    threshold = config["threshold"]

    X_benchmark = X_test_vagifa[features]

    # Preparar registros individuais fora da medição
    amostras = [
        X_benchmark.iloc[[i]]
        for i in indices
    ]

    # Aquecimento individual e em lote
    for i in range(N_AQUECIMENTO):
        inferencia(modelo, amostras[i], threshold)
        inferencia(modelo, X_benchmark, threshold)

    # ======================================
    # Latência individual
    # ======================================

    tempos_individuais = []

    for amostra in amostras:

        inicio = time.perf_counter_ns()

        inferencia(modelo, amostra, threshold)

        fim = time.perf_counter_ns()

        tempos_individuais.append(
            (fim - inicio) / 1e6
        )

    # ======================================
    # Inferência em lote
    # ======================================

    tempos_lote = []

    for _ in range(N_REPETICOES_LOTE):

        inicio = time.perf_counter_ns()

        inferencia(modelo, X_benchmark, threshold)

        fim = time.perf_counter_ns()

        tempos_lote.append(
            (fim - inicio) / 1e9
        )

    # Registros processados por segundo
    throughput = (
        len(X_benchmark) * N_REPETICOES_LOTE
        / np.sum(tempos_lote)
    )

    resultados_desempenho_vagifa.append({
        "Modelo": nome,
        "Features": len(features),
        "Latencia_media_ms": np.mean(tempos_individuais),
        "Latencia_p50_ms": np.percentile(tempos_individuais, 50),
        "Latencia_p95_ms": np.percentile(tempos_individuais, 95),
        "Latencia_p99_ms": np.percentile(tempos_individuais, 99),
        "Latencia_desvio_ms": np.std(tempos_individuais),
        "Tempo_medio_lote_ms": np.mean(tempos_lote) * 1000,
        "Registros_por_lote": len(X_benchmark),
        "Throughput_registros_s": throughput
    })

    print(f"\nBenchmark concluído: {nome}")

# Tabela final
df_desempenho_vagifa = pd.DataFrame(
    resultados_desempenho_vagifa
)

print("\n=== DESEMPENHO COMPUTACIONAL ===")

display(df_desempenho_vagifa.round(4))


=== AMBIENTE COMPUTACIONAL ===
Sistema: Linux-6.6.159+-x86_64-with-glibc2.39
Arquitetura: x86_64
CPUs_logicas: 2
Python: 3.13.18
Scikit_learn: 1.6.1
XGBoost: 3.4.1
NumPy: 2.4.6
Pandas: 2.2.3
CPU: Intel(R) Xeon(R) CPU @ 2.20GHz

Benchmark concluído: Regressao_Logistica

Benchmark concluído: Regressao_Logistica_Balanceada

Benchmark concluído: XGBoost

=== DESEMPENHO COMPUTACIONAL ===


,Modelo,Features,Latencia_media_ms,Latencia_p50_ms,Latencia_p95_ms,Latencia_p99_ms,Latencia_desvio_ms,Tempo_medio_lote_ms,Registros_por_lote,Throughput_registros_s
0,Regressao_Logistica,6,5.4829,2.1771,19.4063,27.8510,6.0380,3.7972,1964,517225.2058
1,Regressao_Logistica_Balanceada,6,2.5156,2.2083,4.9052,8.2243,1.2133,2.2724,1964,864293.4458
2,XGBoost,18,8.3750,4.0374,18.0546,23.6608,15.8208,7.7909,1964,252088.2724


In [45]:

df_comparacao_final_vagifa = (
    df_teste_final_vagifa.merge(
        df_desempenho_vagifa,
        on=["Modelo", "Features"]
    )
)

display(
    df_comparacao_final_vagifa[
        [
            "Modelo",
            "Features",
            "AP",
            "PR_AUC",
            "Precision",
            "Recall",
            "F1",
            "Latencia_media_ms",
            "Latencia_p95_ms",
            "Latencia_p99_ms",
            "Throughput_registros_s"
        ]
    ].round(4)
)

# Exportar os resultados
df_desempenho_vagifa.to_csv(
    pasta / "desempenho_computacional.csv",
    index=False
)

df_comparacao_final_vagifa.to_csv(
    pasta / "comparacao_final_vagifa.csv",
    index=False
)

pd.DataFrame([ambiente]).to_csv(
    pasta / "ambiente_computacional.csv",
    index=False
)


,Modelo,Features,AP,PR_AUC,Precision,Recall,F1,Latencia_media_ms,Latencia_p95_ms,Latencia_p99_ms,Throughput_registros_s
0,Regressao_Logistica,6,0.6185,0.6040,0.5603,0.6284,0.5924,5.4829,19.4063,27.8510,517225.2058
1,Regressao_Logistica_Balanceada,6,0.6257,0.6120,0.5625,0.6193,0.5895,2.5156,4.9052,8.2243,864293.4458
2,XGBoost,18,0.9393,0.9393,0.8485,0.8349,0.8416,8.3750,18.0546,23.6608,252088.2724


In [46]:

import shutil
from google.colab import files

arquivo_zip = shutil.make_archive(
    "/content/resultados_vagifa",
    "zip",
    root_dir="/content/resultados_vagifa"
)

print("Arquivo gerado:", arquivo_zip)

files.download(arquivo_zip)


Arquivo gerado: /content/resultados_vagifa.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>